# 07 — Resumable LLM Batch Inference

## Purpose

This notebook executes the formal paired LLM inference experiment for the
`primary_46` feature set.

Each of the 200 network-flow records is presented to the same
OpenCode-backed `uoa/MiniMax-M3` model under two conditions:

1. structured JSON; and
2. deterministic natural-language text.

The two conditions contain the same underlying feature names and values. Only
their representation differs.

## Locked execution backend

Notebook 06 established that the direct Chat Completions and Responses API
paths exhausted their output allowances before reliably returning complete
research responses. The OpenCode-backed route subsequently passed:

* a minimal connectivity probe;
* a paired research-record preflight; and
* a five-record, ten-request reliability pilot.

Formal inference therefore uses the validated OpenCode configuration:

* OpenCode version 1.18.22;
* model route `uoa/MiniMax-M3`;
* deterministic prompt-template version `0.1.0`;
* `--pure`;
* one empty temporary directory per request;
* no file attachments;
* no automatic permission approval;
* JSON event output;
* a 300-second per-request timeout; and
* no notebook-supplied model-output token limit.

## Inference boundary

This notebook performs inference and operational validation only. It must not
load, inspect or join the private ground-truth file.

The notebook records:

* visible model responses;
* request and prompt hashes;
* OpenCode event and token metadata;
* schema-validation results;
* feature-grounding results; and
* bounded operational errors.

It does not retain API credentials or hidden reasoning content.

Ground truth will be joined only after the complete inference artifact has been
validated and frozen in a later notebook.


In [1]:
from __future__ import annotations

import hashlib
import json
import subprocess
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


def find_project_root(start: Path) -> Path:
    """
    Locate the repository root from either the project or notebooks directory.
    """
    start = start.resolve()

    for candidate in (start, *start.parents):
        if (
            (candidate / "README.md").exists()
            and (candidate / "notebooks").is_dir()
            and (candidate / "configs").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root containing "
        "README.md, notebooks/ and configs/."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

REPRESENTATION_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "representations"
    / "primary_46"
)

STRUCTURED_PATH = (
    REPRESENTATION_DIR / "structured.jsonl"
)

TEXT_PATH = (
    REPRESENTATION_DIR / "deterministic_text.jsonl"
)

EQUIVALENCE_PATH = (
    REPRESENTATION_DIR / "equivalence_validation.csv"
)

REPRESENTATION_MANIFEST_PATH = (
    REPRESENTATION_DIR / "manifest.json"
)

PROTOCOL_PATH = (
    PROJECT_ROOT
    / "configs"
    / "llm_protocol_primary_46.json"
)

OUTPUT_SCHEMA_PATH = (
    PROJECT_ROOT
    / "configs"
    / "llm_output_schema.json"
)

RELIABILITY_CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "results"
    / "reliability"
    / "primary_46"
    / "checkpoints"
)

BATCH_RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "inference"
    / "primary_46"
)

BATCH_CHECKPOINT_DIR = (
    BATCH_RESULTS_DIR / "checkpoints"
)

OPENCODE_EXECUTABLE = Path(
    "/opt/homebrew/bin/opencode"
)

required_paths = [
    STRUCTURED_PATH,
    TEXT_PATH,
    EQUIVALENCE_PATH,
    REPRESENTATION_MANIFEST_PATH,
    PROTOCOL_PATH,
    OUTPUT_SCHEMA_PATH,
    RELIABILITY_CHECKPOINT_DIR,
    OPENCODE_EXECUTABLE,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Required batch-inference inputs are missing:\n- "
        + "\n- ".join(missing_paths)
    )

environment_summary = pd.Series(
    {
        "project_root": str(PROJECT_ROOT),
        "representation_directory": str(
            REPRESENTATION_DIR
        ),
        "reliability_checkpoint_directory": str(
            RELIABILITY_CHECKPOINT_DIR
        ),
        "batch_result_directory": str(
            BATCH_RESULTS_DIR
        ),
        "required_paths_checked": len(
            required_paths
        ),
        "missing_required_paths": len(
            missing_paths
        ),
        "opencode_executable_found": (
            OPENCODE_EXECUTABLE.exists()
        ),
        "ground_truth_path_defined": False,
        "ground_truth_loaded": False,
        "network_request_made": False,
    },
    name="value",
)

environment_summary

project_root                            /Users/ruiwang/Developer/compsci742-rui-pilot
representation_directory            /Users/ruiwang/Developer/compsci742-rui-pilot/...
reliability_checkpoint_directory    /Users/ruiwang/Developer/compsci742-rui-pilot/...
batch_result_directory              /Users/ruiwang/Developer/compsci742-rui-pilot/...
required_paths_checked                                                              8
missing_required_paths                                                              0
opencode_executable_found                                                        True
ground_truth_path_defined                                                       False
ground_truth_loaded                                                             False
network_request_made                                                            False
Name: value, dtype: object

## Load and validate the locked inference inputs

The formal batch uses the representation artifacts and provider-independent
protocol produced before backend selection.

This stage loads:

* 200 structured representations;
* 200 deterministic-text representations;
* the representation-equivalence audit;
* the representation manifest;
* the locked `primary_46` protocol; and
* the locked response schema.

The two representation files must contain the same sample identifiers in the
same order, and each paired record must reference the same canonical-payload
hash. These checks establish that later condition-level requests refer to the
same underlying network-flow observations.

Only model inputs and public experiment configuration are loaded. The private
ground-truth file is neither referenced nor opened.

This stage performs local file reads and assertions only. It makes no network
request.


In [2]:
def read_jsonl(path: Path) -> list[dict]:
    """
    Read a UTF-8 JSON Lines file into a list of dictionaries.
    """
    records = []

    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(
            handle,
            start=1,
        ):
            stripped = line.strip()

            if not stripped:
                continue

            try:
                record = json.loads(stripped)
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSON in {path.name}, "
                    f"line {line_number}."
                ) from exc

            if not isinstance(record, dict):
                raise ValueError(
                    f"Line {line_number} in {path.name} "
                    "is not a JSON object."
                )

            records.append(record)

    return records


def sha256_file(path: Path) -> str:
    """Calculate the SHA-256 digest of one file's exact bytes."""
    return hashlib.sha256(path.read_bytes()).hexdigest()


structured_records = read_jsonl(
    STRUCTURED_PATH
)

text_records = read_jsonl(
    TEXT_PATH
)

equivalence_validation = pd.read_csv(
    EQUIVALENCE_PATH
)

with REPRESENTATION_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    representation_manifest = json.load(handle)

with PROTOCOL_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    llm_protocol = json.load(handle)

with OUTPUT_SCHEMA_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    llm_output_schema = json.load(handle)


structured_sample_ids = [
    record["sample_id"]
    for record in structured_records
]

text_sample_ids = [
    record["sample_id"]
    for record in text_records
]

paired_payload_hashes_match = all(
    structured_record["canonical_payload_sha256"]
    == text_record["canonical_payload_sha256"]
    for structured_record, text_record in zip(
        structured_records,
        text_records,
        strict=True,
    )
)

paired_feature_set_ids_match = all(
    structured_record["feature_set_id"]
    == text_record["feature_set_id"]
    == "primary_46"
    for structured_record, text_record in zip(
        structured_records,
        text_records,
        strict=True,
    )
)

model_inputs_are_non_empty_strings = all(
    isinstance(record["model_input"], str)
    and bool(record["model_input"].strip())
    for record in [
        *structured_records,
        *text_records,
    ]
)

# Formal inference requires exactly one paired representation for each of the
# 200 previously selected evaluation records.
assert len(structured_records) == 200
assert len(text_records) == 200
assert len(equivalence_validation) == 200

assert structured_sample_ids == text_sample_ids
assert len(set(structured_sample_ids)) == 200

assert paired_payload_hashes_match
assert paired_feature_set_ids_match
assert model_inputs_are_non_empty_strings

assert (
    llm_protocol["feature_set"]["feature_count"]
    == 46
)
assert (
    llm_protocol["feature_set"]["feature_set_id"]
    == "primary_46"
)
assert (
    llm_protocol["evaluation_inputs"]["record_count"]
    == 200
)
assert (
    llm_protocol["evaluation_inputs"]["paired_records"]
    is True
)
assert set(
    llm_protocol["evaluation_inputs"]["conditions"]
) == {
    "structured",
    "deterministic_text",
}
assert llm_output_schema["strict"] is True

locked_input_summary = pd.Series(
    {
        "structured_record_count": len(
            structured_records
        ),
        "deterministic_text_record_count": len(
            text_records
        ),
        "equivalence_validation_rows": len(
            equivalence_validation
        ),
        "sample_order_matches": (
            structured_sample_ids
            == text_sample_ids
        ),
        "unique_sample_ids": len(
            set(structured_sample_ids)
        ),
        "paired_payload_hashes_match": (
            paired_payload_hashes_match
        ),
        "paired_feature_set_ids_match": (
            paired_feature_set_ids_match
        ),
        "model_inputs_are_non_empty_strings": (
            model_inputs_are_non_empty_strings
        ),
        "feature_set_id": (
            llm_protocol["feature_set"][
                "feature_set_id"
            ]
        ),
        "feature_count": (
            llm_protocol["feature_set"][
                "feature_count"
            ]
        ),
        "structured_file_sha256": sha256_file(
            STRUCTURED_PATH
        ),
        "text_file_sha256": sha256_file(
            TEXT_PATH
        ),
        "protocol_file_sha256": sha256_file(
            PROTOCOL_PATH
        ),
        "output_schema_file_sha256": sha256_file(
            OUTPUT_SCHEMA_PATH
        ),
        "ground_truth_loaded": False,
        "network_request_made": False,
    },
    name="value",
)

locked_input_summary

structured_record_count                                                             200
deterministic_text_record_count                                                     200
equivalence_validation_rows                                                         200
sample_order_matches                                                               True
unique_sample_ids                                                                   200
paired_payload_hashes_match                                                        True
paired_feature_set_ids_match                                                       True
model_inputs_are_non_empty_strings                                                 True
feature_set_id                                                               primary_46
feature_count                                                                        46
structured_file_sha256                a74fdbb25e8d24fb9afd06d13982f19689a04685436a65...
text_file_sha256                

## Validate reusable reliability checkpoints

Ten condition-level responses were completed during Notebook 06 under the same
`primary_46` representations and validated OpenCode prompt template intended
for formal inference.

Reusing these responses avoids unnecessary duplicate model calls, but reuse is
authorised only if the stored evidence matches the current locked inputs and
execution configuration.

This stage verifies:

* the reliability manifest authorises batch inference;
* protocol and output-schema file hashes match the current files;
* the OpenCode version, model route and prompt-template version match;
* exactly ten unique record-condition checkpoints are present;
* every stored sample identifier and record index match the current input;
* every canonical-payload hash matches the current paired record;
* every response completed and passed local validation; and
* no checkpoint contains or reports loaded ground truth.

This stage reads existing result artifacts only. It makes no external request.


In [3]:
RELIABILITY_MANIFEST_PATH = (
    PROJECT_ROOT
    / "results"
    / "reliability"
    / "primary_46"
    / "opencode_reliability_manifest.json"
)

if not RELIABILITY_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Missing reliability manifest: "
        f"{RELIABILITY_MANIFEST_PATH}"
    )

with RELIABILITY_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    reliability_manifest = json.load(handle)

reliability_checkpoint_paths = sorted(
    RELIABILITY_CHECKPOINT_DIR.glob("*.json")
)

reliability_checkpoints = []

for checkpoint_path in reliability_checkpoint_paths:
    with checkpoint_path.open(
        "r",
        encoding="utf-8",
    ) as handle:
        checkpoint = json.load(handle)

    if not isinstance(checkpoint, dict):
        raise ValueError(
            f"Checkpoint is not a JSON object: "
            f"{checkpoint_path}"
        )

    reliability_checkpoints.append(checkpoint)


# Verify the aggregate reliability decision against the files currently loaded
# for formal inference.
assert (
    reliability_manifest["validation"][
        "batch_inference_authorised"
    ]
    is True
)

assert (
    reliability_manifest["protocol"]["sha256"]
    == sha256_file(PROTOCOL_PATH)
)

assert (
    reliability_manifest["output_schema"]["sha256"]
    == sha256_file(OUTPUT_SCHEMA_PATH)
)

assert (
    reliability_manifest["feature_set"]["id"]
    == "primary_46"
)

assert (
    reliability_manifest["feature_set"]["feature_count"]
    == 46
)

assert (
    reliability_manifest["backend"]["name"]
    == "opencode"
)

assert (
    reliability_manifest["backend"]["version"]
    == "1.18.22"
)

assert (
    reliability_manifest["backend"]["model_route"]
    == "uoa/MiniMax-M3"
)

assert (
    reliability_manifest["backend"][
        "prompt_template_version"
    ]
    == "0.1.0"
)


# Exactly five records and two representation conditions were validated in
# Notebook 06.
expected_reliability_indices = {
    0,
    50,
    100,
    150,
    199,
}

expected_conditions = {
    "structured",
    "deterministic_text",
}

assert len(reliability_checkpoints) == 10

checkpoint_keys = [
    (
        checkpoint["record_index"],
        checkpoint["condition"],
    )
    for checkpoint in reliability_checkpoints
]

assert len(set(checkpoint_keys)) == 10

assert {
    checkpoint["record_index"]
    for checkpoint in reliability_checkpoints
} == expected_reliability_indices

assert {
    checkpoint["condition"]
    for checkpoint in reliability_checkpoints
} == expected_conditions


# Match every checkpoint back to the currently loaded paired input record.
for checkpoint in reliability_checkpoints:
    record_index = checkpoint["record_index"]
    condition = checkpoint["condition"]

    structured_record = structured_records[record_index]
    text_record = text_records[record_index]

    expected_sample_id = structured_record["sample_id"]
    expected_payload_sha256 = structured_record[
        "canonical_payload_sha256"
    ]

    assert (
        expected_sample_id
        == text_record["sample_id"]
    )

    assert (
        expected_payload_sha256
        == text_record["canonical_payload_sha256"]
    )

    assert (
        checkpoint["sample_id"]
        == expected_sample_id
    )

    assert (
        checkpoint["canonical_payload_sha256"]
        == expected_payload_sha256
    )

    assert checkpoint["condition"] == condition
    assert checkpoint["feature_set_id"] == "primary_46"
    assert checkpoint["backend"] == "opencode"
    assert checkpoint["opencode_version"] == "1.18.22"
    assert checkpoint["requested_model"] == "uoa/MiniMax-M3"

    assert (
        checkpoint["prompt_template_version"]
        == "0.1.0"
    )

    assert checkpoint["request_completed"] is True
    assert checkpoint["return_code"] == 0
    assert checkpoint["visible_json_parsed"] is True
    assert checkpoint["schema_structure_valid"] is True
    assert checkpoint["grounding_valid"] is True
    assert checkpoint["overall_response_valid"] is True
    assert checkpoint["budget_exhausted"] is False
    assert checkpoint["ground_truth_loaded"] is False


reusable_checkpoint_summary = pd.Series(
    {
        "reliability_manifest_found": (
            RELIABILITY_MANIFEST_PATH.exists()
        ),
        "protocol_hash_matches": (
            reliability_manifest["protocol"]["sha256"]
            == sha256_file(PROTOCOL_PATH)
        ),
        "output_schema_hash_matches": (
            reliability_manifest["output_schema"]["sha256"]
            == sha256_file(OUTPUT_SCHEMA_PATH)
        ),
        "backend": (
            reliability_manifest["backend"]["name"]
        ),
        "opencode_version": (
            reliability_manifest["backend"]["version"]
        ),
        "model_route": (
            reliability_manifest["backend"][
                "model_route"
            ]
        ),
        "prompt_template_version": (
            reliability_manifest["backend"][
                "prompt_template_version"
            ]
        ),
        "expected_reusable_checkpoint_count": 10,
        "observed_reusable_checkpoint_count": len(
            reliability_checkpoints
        ),
        "unique_checkpoint_keys": len(
            set(checkpoint_keys)
        ),
        "all_requests_completed": all(
            checkpoint["request_completed"]
            for checkpoint in reliability_checkpoints
        ),
        "all_responses_valid": all(
            checkpoint["overall_response_valid"]
            for checkpoint in reliability_checkpoints
        ),
        "all_payload_hashes_match_current_inputs": all(
            checkpoint["canonical_payload_sha256"]
            == structured_records[
                checkpoint["record_index"]
            ]["canonical_payload_sha256"]
            for checkpoint in reliability_checkpoints
        ),
        "batch_inference_authorised": (
            reliability_manifest["validation"][
                "batch_inference_authorised"
            ]
        ),
        "ground_truth_loaded": False,
        "network_request_made": False,
    },
    name="value",
)

reusable_checkpoint_summary

reliability_manifest_found                           True
protocol_hash_matches                                True
output_schema_hash_matches                           True
backend                                          opencode
opencode_version                                  1.18.22
model_route                                uoa/MiniMax-M3
prompt_template_version                             0.1.0
expected_reusable_checkpoint_count                     10
observed_reusable_checkpoint_count                     10
unique_checkpoint_keys                                 10
all_requests_completed                               True
all_responses_valid                                  True
all_payload_hashes_match_current_inputs              True
batch_inference_authorised                           True
ground_truth_loaded                                 False
network_request_made                                False
Name: value, dtype: object

## Construct the complete paired request plan

The formal inference plan contains 400 condition-level requests: two
representations for each of 200 paired network-flow records.

The prompt for every request is rebuilt from the currently loaded locked
protocol, response schema and representation artifact. The resulting prompt
hash is compared with any reusable Notebook 06 checkpoint. Reuse is permitted
only when the regenerated prompt hash and all record identifiers match exactly.

To reduce systematic order effects, representation order is counterbalanced by
record index:

* even-numbered record indices schedule structured first;
* odd-numbered record indices schedule deterministic text first.

This produces 100 records in each ordering group. Both conditions for a record
retain the same underlying canonical payload, model route, prompt template and
validation policy.

The request plan identifies already completed and pending requests but does not
execute either group. Ground truth remains unloaded.


In [4]:
OPENCODE_VERSION = "1.18.22"
OPENCODE_MODEL_ROUTE = "uoa/MiniMax-M3"
OPENCODE_PROMPT_TEMPLATE_VERSION = "0.1.0"
OPENCODE_REQUEST_TIMEOUT_SECONDS = 300

COMMON_SYSTEM_INSTRUCTION = (
    llm_protocol["instructions"]["system_instruction"]
)

RECORD_OPENING_DELIMITER = (
    llm_protocol["instructions"][
        "record_opening_delimiter"
    ]
)

RECORD_CLOSING_DELIMITER = (
    llm_protocol["instructions"][
        "record_closing_delimiter"
    ]
)

OPENCODE_RESEARCH_SCHEMA_TEXT = json.dumps(
    llm_output_schema["schema"],
    ensure_ascii=False,
    sort_keys=True,
    separators=(",", ":"),
)


def stable_json_sha256(value: object) -> str:
    """
    Calculate a deterministic SHA-256 digest for a JSON-compatible object.
    """
    canonical_json = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )

    return hashlib.sha256(
        canonical_json.encode("utf-8")
    ).hexdigest()


def build_user_message(model_input: str) -> str:
    """
    Enclose one validated representation in the locked record delimiters.

    No sample identifier, label, dataset name, class balance, threshold or
    worked example is introduced.
    """
    if not isinstance(model_input, str):
        raise TypeError(
            "model_input must be a string."
        )

    if not model_input.strip():
        raise ValueError(
            "model_input must not be empty."
        )

    if (
        RECORD_OPENING_DELIMITER in model_input
        or RECORD_CLOSING_DELIMITER in model_input
    ):
        raise ValueError(
            "model_input contains a reserved delimiter."
        )

    return (
        f"{RECORD_OPENING_DELIMITER}\n"
        f"{model_input}\n"
        f"{RECORD_CLOSING_DELIMITER}"
    )


def build_logical_request_contract(
    model_input: str,
) -> dict:
    """
    Build the provider-independent request components for one representation.
    """
    return {
        "system_instruction": COMMON_SYSTEM_INSTRUCTION,
        "user_message": build_user_message(
            model_input
        ),
        "output_schema": llm_output_schema,
    }


def build_opencode_research_prompt(
    request_contract: dict,
) -> str:
    """
    Convert a logical request into the locked single-prompt OpenCode format.

    This implementation must remain byte-for-byte compatible with prompt
    template version 0.1.0 validated in Notebook 06.
    """
    required_fields = {
        "system_instruction",
        "user_message",
        "output_schema",
    }

    if set(request_contract) != required_fields:
        raise ValueError(
            "Unexpected logical request fields."
        )

    if request_contract["output_schema"] != llm_output_schema:
        raise ValueError(
            "The request does not use the locked output schema."
        )

    return (
        "<research_instruction>\n"
        f"{request_contract['system_instruction']}\n"
        "</research_instruction>\n\n"
        "<required_output_json_schema>\n"
        f"{OPENCODE_RESEARCH_SCHEMA_TEXT}\n"
        "</required_output_json_schema>\n\n"
        "<research_input>\n"
        f"{request_contract['user_message']}\n"
        "</research_input>\n\n"
        "Follow the research instruction using only the enclosed research "
        "input. Return only the JSON object required by the enclosed schema."
    )


reusable_checkpoints_by_key = {
    (
        checkpoint["record_index"],
        checkpoint["condition"],
    ): checkpoint
    for checkpoint in reliability_checkpoints
}

assert len(reusable_checkpoints_by_key) == 10

batch_request_plan = []

for record_index, (
    structured_record,
    text_record,
) in enumerate(
    zip(
        structured_records,
        text_records,
        strict=True,
    )
):
    assert (
        structured_record["sample_id"]
        == text_record["sample_id"]
    )
    assert (
        structured_record["canonical_payload_sha256"]
        == text_record["canonical_payload_sha256"]
    )

    sample_id = structured_record["sample_id"]
    canonical_payload_sha256 = structured_record[
        "canonical_payload_sha256"
    ]

    records_by_condition = {
        "structured": structured_record,
        "deterministic_text": text_record,
    }

    # Alternate which representation is scheduled first for each record.
    if record_index % 2 == 0:
        condition_order = [
            "structured",
            "deterministic_text",
        ]
        order_group = "structured_first"
    else:
        condition_order = [
            "deterministic_text",
            "structured",
        ]
        order_group = "deterministic_text_first"

    for within_pair_position, condition in enumerate(
        condition_order,
        start=1,
    ):
        source_record = records_by_condition[
            condition
        ]

        logical_contract = (
            build_logical_request_contract(
                source_record["model_input"]
            )
        )

        prompt = build_opencode_research_prompt(
            logical_contract
        )

        prompt_sha256 = stable_json_sha256(
            {
                "template_version": (
                    OPENCODE_PROMPT_TEMPLATE_VERSION
                ),
                "prompt": prompt,
            }
        )

        request_key = (
            record_index,
            condition,
        )

        reusable_checkpoint = (
            reusable_checkpoints_by_key.get(
                request_key
            )
        )

        if reusable_checkpoint is not None:
            # This is the strongest reuse check: Notebook 07 independently
            # rebuilds the complete prompt and requires its digest to match the
            # original Notebook 06 checkpoint.
            assert (
                reusable_checkpoint["sample_id"]
                == sample_id
            )
            assert (
                reusable_checkpoint[
                    "canonical_payload_sha256"
                ]
                == canonical_payload_sha256
            )
            assert (
                reusable_checkpoint["prompt_sha256"]
                == prompt_sha256
            )
            assert (
                reusable_checkpoint[
                    "overall_response_valid"
                ]
                is True
            )

        batch_request_plan.append(
            {
                "request_plan_position": len(
                    batch_request_plan
                ),
                "record_index": record_index,
                "sample_id": sample_id,
                "condition": condition,
                "order_group": order_group,
                "within_pair_position": (
                    within_pair_position
                ),
                "canonical_payload_sha256": (
                    canonical_payload_sha256
                ),
                "prompt": prompt,
                "prompt_sha256": prompt_sha256,
                "reusable_checkpoint_available": (
                    reusable_checkpoint is not None
                ),
            }
        )


assert len(batch_request_plan) == 400

batch_request_keys = [
    (
        request["record_index"],
        request["condition"],
    )
    for request in batch_request_plan
]

assert len(set(batch_request_keys)) == 400

assert sum(
    request["reusable_checkpoint_available"]
    for request in batch_request_plan
) == 10

assert sum(
    request["order_group"]
    == "structured_first"
    and request["within_pair_position"] == 1
    for request in batch_request_plan
) == 100

assert sum(
    request["order_group"]
    == "deterministic_text_first"
    and request["within_pair_position"] == 1
    for request in batch_request_plan
) == 100

assert all(
    request["sample_id"]
    not in request["prompt"]
    for request in batch_request_plan
)

pending_batch_requests = [
    request
    for request in batch_request_plan
    if not request[
        "reusable_checkpoint_available"
    ]
]

batch_plan_preview = pd.DataFrame(
    [
        {
            "request_plan_position": request[
                "request_plan_position"
            ],
            "record_index": request[
                "record_index"
            ],
            "sample_id": request["sample_id"],
            "condition": request["condition"],
            "order_group": request["order_group"],
            "within_pair_position": request[
                "within_pair_position"
            ],
            "prompt_characters": len(
                request["prompt"]
            ),
            "prompt_sha256": request[
                "prompt_sha256"
            ],
            "reusable_checkpoint_available": request[
                "reusable_checkpoint_available"
            ],
        }
        for request in batch_request_plan
    ]
)

display(batch_plan_preview.head(12))

batch_plan_summary = pd.Series(
    {
        "paired_record_count": len(
            structured_records
        ),
        "planned_request_count": len(
            batch_request_plan
        ),
        "unique_request_keys": len(
            set(batch_request_keys)
        ),
        "structured_request_count": sum(
            request["condition"] == "structured"
            for request in batch_request_plan
        ),
        "deterministic_text_request_count": sum(
            request["condition"]
            == "deterministic_text"
            for request in batch_request_plan
        ),
        "structured_first_pair_count": 100,
        "deterministic_text_first_pair_count": 100,
        "reusable_checkpoint_count": sum(
            request[
                "reusable_checkpoint_available"
            ]
            for request in batch_request_plan
        ),
        "pending_request_count": len(
            pending_batch_requests
        ),
        "all_reusable_prompt_hashes_match": all(
            reusable_checkpoints_by_key[
                (
                    request["record_index"],
                    request["condition"],
                )
            ]["prompt_sha256"]
            == request["prompt_sha256"]
            for request in batch_request_plan
            if request[
                "reusable_checkpoint_available"
            ]
        ),
        "sample_ids_sent_to_model": False,
        "ground_truth_loaded": False,
        "network_request_made": False,
    },
    name="value",
)

batch_plan_summary

,request_plan_position,record_index,sample_id,condition,order_group,within_pair_position,prompt_characters,prompt_sha256,reusable_checkpoint_available
0,0,0,pilot_001,structured,structured_first,1,4049,f49c1346affe94d3a4be85086cfc5f362c91bcbbfb992d...,True
1,1,0,pilot_001,deterministic_text,structured_first,2,4026,577d3214b51eed9805ffb1291cc5643c6711e78b0f2718...,True
2,2,1,pilot_002,deterministic_text,deterministic_text_first,1,4029,19a36efbe748bb82e0686ae8da398539329a7fe3ae2687...,False
3,3,1,pilot_002,structured,deterministic_text_first,2,4052,23871ab075007d7f05e2934d4d1c631b86a824054209d7...,False
4,4,2,pilot_003,structured,structured_first,1,4025,bde8c80e275a053c700b034d56b1ebe0ae4c7065c1653d...,False
5,5,2,pilot_003,deterministic_text,structured_first,2,4002,92e997c9f052e5641ee15b5a0f12e625c5eacf0464c933...,False
6,6,3,pilot_004,deterministic_text,deterministic_text_first,1,4027,2b66ce380b7bfbb994d1b9f51f19086fee282e1de7e641...,False
7,7,3,pilot_004,structured,deterministic_text_first,2,4050,80891e1295fd4637f904e27dd5b9f874d4eb5947692500...,False
8,8,4,pilot_005,structured,structured_first,1,4054,8b8eddce220d5be0d7155a5ad5adb3d4c055540f72d7d4...,False
9,9,4,pilot_005,deterministic_text,structured_first,2,4031,8461a158538303d2e0c58ea34e9cd3d809d4fdb09e7210...,False


paired_record_count                      200
planned_request_count                    400
unique_request_keys                      400
structured_request_count                 200
deterministic_text_request_count         200
structured_first_pair_count              100
deterministic_text_first_pair_count      100
reusable_checkpoint_count                 10
pending_request_count                    390
all_reusable_prompt_hashes_match        True
sample_ids_sent_to_model               False
ground_truth_loaded                    False
network_request_made                   False
Name: value, dtype: object

## Seed the formal batch checkpoint store

The ten validated responses produced during backend reliability testing are part of the same locked experiment. Their sample IDs, conditions, canonical payload hashes, prompt hashes, protocol hash, output-schema hash, backend version, model route, and prompt-template version have already been verified against the current batch configuration.

This section reuses those responses by copying their checkpoint files into the formal batch checkpoint directory.

To preserve the original evidence:

* checkpoint files are copied byte-for-byte;
* no model request is repeated;
* existing destination files are never silently overwritten;
* an existing destination is accepted only when its SHA-256 hash matches the source file; and
* the copied checkpoint is validated again against the corresponding batch-plan entry.

This operation does not load ground-truth labels and does not make any network request.


In [5]:
def batch_checkpoint_path(record_index: int, condition: str) -> Path:
    """Return the canonical formal-batch checkpoint path for one request."""
    return (
        BATCH_CHECKPOINT_DIR
        / f"record_{record_index:04d}__{condition}.json"
    )


def copy_checkpoint_without_overwrite(
    source_path: Path,
    destination_path: Path,
) -> dict:
    """
    Copy one checkpoint byte-for-byte without silently overwriting evidence.

    If the destination already exists, it is reused only when its SHA-256
    hash exactly matches the source checkpoint.
    """
    source_bytes = source_path.read_bytes()
    source_sha256 = hashlib.sha256(source_bytes).hexdigest()

    if destination_path.exists():
        destination_sha256 = sha256_file(destination_path)

        if destination_sha256 != source_sha256:
            raise RuntimeError(
                "Existing batch checkpoint differs from the validated "
                f"reliability checkpoint: {destination_path}"
            )

        return {
            "checkpoint_action": "verified_existing",
            "source_sha256": source_sha256,
            "destination_sha256": destination_sha256,
        }

    destination_path.parent.mkdir(parents=True, exist_ok=True)

    # Write to a temporary file first so that an interrupted write cannot
    # leave a partially written checkpoint at the canonical destination.
    temporary_path = destination_path.with_suffix(
        destination_path.suffix + ".tmp"
    )

    if temporary_path.exists():
        raise RuntimeError(
            f"Unexpected temporary checkpoint already exists: {temporary_path}"
        )

    temporary_path.write_bytes(source_bytes)

    temporary_sha256 = sha256_file(temporary_path)
    if temporary_sha256 != source_sha256:
        raise RuntimeError(
            f"Temporary checkpoint failed SHA-256 verification: {temporary_path}"
        )

    # Path.replace is atomic when the source and destination are on the
    # same filesystem. The destination was already confirmed absent.
    temporary_path.replace(destination_path)

    destination_sha256 = sha256_file(destination_path)
    if destination_sha256 != source_sha256:
        raise RuntimeError(
            f"Final checkpoint failed SHA-256 verification: {destination_path}"
        )

    return {
        "checkpoint_action": "copied",
        "source_sha256": source_sha256,
        "destination_sha256": destination_sha256,
    }


BATCH_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

reusable_checkpoint_actions = []

for plan_entry in batch_request_plan:
    if not plan_entry["reusable_checkpoint_available"]:
        continue

    record_index = int(plan_entry["record_index"])
    condition = plan_entry["condition"]

    source_path = (
        RELIABILITY_CHECKPOINT_DIR
        / f"record_{record_index:04d}__{condition}.json"
    )
    destination_path = batch_checkpoint_path(record_index, condition)

    if not source_path.is_file():
        raise FileNotFoundError(
            f"Validated reusable checkpoint is missing: {source_path}"
        )

    # Revalidate the source checkpoint against the locked batch-plan entry
    # before admitting it into the formal checkpoint store.
    source_checkpoint = json.loads(source_path.read_text(encoding="utf-8"))

    assert source_checkpoint["sample_id"] == plan_entry["sample_id"]
    assert source_checkpoint["condition"] == condition
    assert (
        source_checkpoint["canonical_payload_sha256"]
        == plan_entry["canonical_payload_sha256"]
    )
    assert source_checkpoint["prompt_sha256"] == plan_entry["prompt_sha256"]
    assert source_checkpoint["request_completed"] is True
    assert source_checkpoint["overall_response_valid"] is True
    assert source_checkpoint.get("ground_truth_loaded", False) is False

    copy_result = copy_checkpoint_without_overwrite(
        source_path=source_path,
        destination_path=destination_path,
    )

    # Validate the formal copy independently after it reaches its final path.
    destination_checkpoint = json.loads(
        destination_path.read_text(encoding="utf-8")
    )

    assert destination_checkpoint == source_checkpoint
    assert sha256_file(destination_path) == sha256_file(source_path)

    reusable_checkpoint_actions.append(
        {
            "record_index": record_index,
            "sample_id": plan_entry["sample_id"],
            "condition": condition,
            "checkpoint_action": copy_result["checkpoint_action"],
            "source_sha256": copy_result["source_sha256"],
            "destination_sha256": copy_result["destination_sha256"],
            "exact_byte_hash_match": (
                copy_result["source_sha256"]
                == copy_result["destination_sha256"]
            ),
            "ground_truth_loaded": False,
            "network_request_made": False,
        }
    )


reusable_checkpoint_action_table = pd.DataFrame(
    reusable_checkpoint_actions
).sort_values(["record_index", "condition"]).reset_index(drop=True)

assert len(reusable_checkpoint_action_table) == 10
assert reusable_checkpoint_action_table["exact_byte_hash_match"].all()

display(reusable_checkpoint_action_table)

checkpoint_seeding_summary = pd.Series(
    {
        "reusable_checkpoints_expected": 10,
        "reusable_checkpoints_present_in_batch": len(
            reusable_checkpoint_action_table
        ),
        "checkpoints_copied": int(
            (
                reusable_checkpoint_action_table["checkpoint_action"]
                == "copied"
            ).sum()
        ),
        "existing_checkpoints_verified": int(
            (
                reusable_checkpoint_action_table["checkpoint_action"]
                == "verified_existing"
            ).sum()
        ),
        "all_exact_byte_hashes_match": bool(
            reusable_checkpoint_action_table[
                "exact_byte_hash_match"
            ].all()
        ),
        "ground_truth_loaded": False,
        "network_request_made": False,
    },
    name="value",
)

checkpoint_seeding_summary

,record_index,sample_id,condition,checkpoint_action,source_sha256,destination_sha256,exact_byte_hash_match,ground_truth_loaded,network_request_made
0,0,pilot_001,deterministic_text,copied,61f245bfc503e302418f315eb2bdb7bb2d2e893151c9f5...,61f245bfc503e302418f315eb2bdb7bb2d2e893151c9f5...,True,False,False
1,0,pilot_001,structured,copied,d4ac9682779436827cfe2e49564492ed9f59988eeccb0d...,d4ac9682779436827cfe2e49564492ed9f59988eeccb0d...,True,False,False
2,50,pilot_051,deterministic_text,copied,184564bde99ab7e5e3262aa7b959008547457659f95412...,184564bde99ab7e5e3262aa7b959008547457659f95412...,True,False,False
3,50,pilot_051,structured,copied,fa34b4dcd2225b927516af69ff2b14fbca23d387af3dde...,fa34b4dcd2225b927516af69ff2b14fbca23d387af3dde...,True,False,False
4,100,pilot_101,deterministic_text,copied,c45775b75c10792d74906d5a10771b1d98ddc139dbabe7...,c45775b75c10792d74906d5a10771b1d98ddc139dbabe7...,True,False,False
5,100,pilot_101,structured,copied,64e6966ab2b8407d1612b55e4a88c6e18dd9138ac36774...,64e6966ab2b8407d1612b55e4a88c6e18dd9138ac36774...,True,False,False
6,150,pilot_151,deterministic_text,copied,44f80b202243c055a227f33d8bd7792045574a6d564a65...,44f80b202243c055a227f33d8bd7792045574a6d564a65...,True,False,False
7,150,pilot_151,structured,copied,7c44f327ec2ac4f45250ede61083b77a108ef430855db3...,7c44f327ec2ac4f45250ede61083b77a108ef430855db3...,True,False,False
8,199,pilot_200,deterministic_text,copied,6cc899608ac3012a2afc4eaea364f7227f61d309f244aa...,6cc899608ac3012a2afc4eaea364f7227f61d309f244aa...,True,False,False
9,199,pilot_200,structured,copied,e2722190585706a8f0ce38dc859bb0aa524449b10fc50c...,e2722190585706a8f0ce38dc859bb0aa524449b10fc50c...,True,False,False


reusable_checkpoints_expected               10
reusable_checkpoints_present_in_batch       10
checkpoints_copied                          10
existing_checkpoints_verified                0
all_exact_byte_hashes_match               True
ground_truth_loaded                      False
network_request_made                     False
Name: value, dtype: object

## Define response parsing and validation helpers

Each OpenCode request returns a JSON event stream rather than only the model's visible answer. The formal batch runner must therefore distinguish between:

1. OpenCode execution metadata;
2. the model's visible response;
3. the JSON object contained in that response; and
4. the validation result applied to the parsed object.

The helpers below implement the same validation layers used during backend preflight and reliability testing:

* successful process completion;
* extraction of exactly one non-empty visible response;
* parsing of the visible response as a JSON object;
* validation against the locked output schema;
* validation of cited feature names against the supplied record; and
* rejection of unsupported or malformed feature citations.

Ground-truth labels are not required for any of these checks. These helpers only assess whether a response is technically usable and grounded in the information supplied to the model.

Defining the helpers does not execute OpenCode and does not make a network request.


In [6]:
from jsonschema import Draft202012Validator


# Compile the locked schema once and reuse the validator for every response.
# This prevents validation rules from changing between requests.
Draft202012Validator.check_schema(llm_output_schema)
LLM_OUTPUT_VALIDATOR = Draft202012Validator(llm_output_schema)


def parse_opencode_event_stream(stdout_text: str) -> list[dict]:
    """
    Parse OpenCode's newline-delimited JSON event stream.

    Blank lines are ignored. Every remaining line must be a valid JSON
    object so that truncated or contaminated command output is not silently
    accepted.
    """
    events = []

    for line_number, raw_line in enumerate(stdout_text.splitlines(), start=1):
        stripped_line = raw_line.strip()

        if not stripped_line:
            continue

        try:
            event = json.loads(stripped_line)
        except json.JSONDecodeError as exc:
            raise ValueError(
                "OpenCode stdout contained a non-JSON event on "
                f"line {line_number}: {exc}"
            ) from exc

        if not isinstance(event, dict):
            raise ValueError(
                f"OpenCode event on line {line_number} is not a JSON object."
            )

        events.append(event)

    if not events:
        raise ValueError("OpenCode returned no parseable JSON events.")

    return events


def extract_visible_response_from_events(events: list[dict]) -> str:
    """
    Reconstruct the model-visible response from OpenCode text events.

    OpenCode may emit several event types. Only events whose type is 'text'
    contribute to the visible model answer.
    """
    visible_text_parts = []

    for event in events:
        if event.get("type") != "text":
            continue

        part = event.get("part", {})
        text_value = part.get("text")

        if isinstance(text_value, str) and text_value:
            visible_text_parts.append(text_value)

    visible_response = "".join(visible_text_parts).strip()

    if not visible_response:
        raise ValueError(
            "No non-empty visible response was found in OpenCode text events."
        )

    return visible_response


def parse_visible_json_object(visible_response: str) -> dict:
    """
    Parse the model-visible response as exactly one JSON object.

    Markdown code fences and additional prose are intentionally rejected
    because the locked protocol requires a bare JSON object.
    """
    try:
        parsed_response = json.loads(visible_response)
    except json.JSONDecodeError as exc:
        raise ValueError(
            f"The visible response is not valid JSON: {exc}"
        ) from exc

    if not isinstance(parsed_response, dict):
        raise ValueError(
            "The visible response must parse to one JSON object."
        )

    return parsed_response


def validate_schema_structure(parsed_response: dict) -> tuple[bool, list[str]]:
    """
    Validate one parsed response against the locked JSON schema.

    All schema violations are returned in a stable order for auditable
    checkpoint records.
    """
    schema_errors = sorted(
        LLM_OUTPUT_VALIDATOR.iter_errors(parsed_response),
        key=lambda error: list(error.absolute_path),
    )

    formatted_errors = []

    for error in schema_errors:
        error_path = ".".join(str(item) for item in error.absolute_path)
        location = error_path if error_path else "<root>"
        formatted_errors.append(f"{location}: {error.message}")

    return len(formatted_errors) == 0, formatted_errors


def extract_cited_feature_names(parsed_response: dict) -> list[str]:
    """
    Extract cited feature names from a schema-valid model response.

    The locked schema is expected to represent each citation either as a
    feature-name string or as an object containing a feature-name field.
    Supporting both forms keeps this extraction helper explicit while the
    JSON-schema validator remains the authoritative structure check.
    """
    cited_features = parsed_response.get("cited_features", [])
    extracted_names = []

    if not isinstance(cited_features, list):
        return extracted_names

    accepted_name_keys = (
        "feature_name",
        "feature",
        "name",
    )

    for citation in cited_features:
        if isinstance(citation, str):
            extracted_names.append(citation)
            continue

        if isinstance(citation, dict):
            feature_name = next(
                (
                    citation[key]
                    for key in accepted_name_keys
                    if isinstance(citation.get(key), str)
                ),
                None,
            )

            if feature_name is not None:
                extracted_names.append(feature_name)

    return extracted_names


def validate_feature_grounding(
    parsed_response: dict,
    canonical_payload: dict,
) -> tuple[bool, dict]:
    """
    Check whether every cited feature belongs to the supplied record.

    This is a record-grounding check, not a claim that the explanation is
    causally or objectively correct. Attribution-reference agreement is
    evaluated later, after inference is complete.
    """
    supplied_features = canonical_payload.get("features")

    if not isinstance(supplied_features, dict):
        raise ValueError(
            "The canonical payload does not contain a 'features' object."
        )

    supplied_feature_names = set(supplied_features)
    cited_feature_names = extract_cited_feature_names(parsed_response)

    invalid_feature_names = sorted(
        {
            feature_name
            for feature_name in cited_feature_names
            if feature_name not in supplied_feature_names
        }
    )

    duplicate_feature_names = sorted(
        {
            feature_name
            for feature_name in cited_feature_names
            if cited_feature_names.count(feature_name) > 1
        }
    )

    grounding_details = {
        "cited_feature_count": len(cited_feature_names),
        "cited_feature_names": cited_feature_names,
        "invalid_feature_names": invalid_feature_names,
        "duplicate_feature_names": duplicate_feature_names,
        "all_cited_features_supplied": len(invalid_feature_names) == 0,
    }

    grounding_valid = (
        len(cited_feature_names) > 0
        and len(invalid_feature_names) == 0
        and len(duplicate_feature_names) == 0
    )

    return grounding_valid, grounding_details


def validate_model_response(
    visible_response: str,
    canonical_payload: dict,
) -> dict:
    """
    Apply parsing, schema, and grounding checks to one visible response.

    Validation failures are returned as structured metadata so the batch
    runner can preserve unsuccessful responses without losing evidence.
    """
    validation_result = {
        "visible_json_parsed": False,
        "schema_structure_valid": False,
        "schema_validation_errors": [],
        "grounding_valid": False,
        "grounding_details": None,
        "overall_response_valid": False,
        "parsed_response": None,
        "error_type": None,
        "error_message": None,
    }

    try:
        parsed_response = parse_visible_json_object(visible_response)
        validation_result["visible_json_parsed"] = True
        validation_result["parsed_response"] = parsed_response

        schema_valid, schema_errors = validate_schema_structure(
            parsed_response
        )
        validation_result["schema_structure_valid"] = schema_valid
        validation_result["schema_validation_errors"] = schema_errors

        # Grounding is evaluated only after the response satisfies the
        # locked structure; malformed responses remain preserved as invalid.
        if schema_valid:
            grounding_valid, grounding_details = validate_feature_grounding(
                parsed_response=parsed_response,
                canonical_payload=canonical_payload,
            )
            validation_result["grounding_valid"] = grounding_valid
            validation_result["grounding_details"] = grounding_details

        validation_result["overall_response_valid"] = bool(
            validation_result["visible_json_parsed"]
            and validation_result["schema_structure_valid"]
            and validation_result["grounding_valid"]
        )

    except Exception as exc:
        validation_result["error_type"] = type(exc).__name__
        validation_result["error_message"] = str(exc)

    return validation_result


validation_helper_summary = pd.Series(
    {
        "locked_schema_compiled": True,
        "event_stream_parser_defined": True,
        "visible_response_extractor_defined": True,
        "schema_validator_defined": True,
        "feature_grounding_validator_defined": True,
        "ground_truth_required": False,
        "ground_truth_loaded": False,
        "network_request_made": False,
    },
    name="value",
)

validation_helper_summary

locked_schema_compiled                  True
event_stream_parser_defined             True
visible_response_extractor_defined      True
schema_validator_defined                True
feature_grounding_validator_defined     True
ground_truth_required                  False
ground_truth_loaded                    False
network_request_made                   False
Name: value, dtype: bool

## Regression-test the batch validators against validated checkpoints

Before executing any new batch request, the response-validation helpers are tested against the ten previously validated reliability checkpoints.

For each reusable checkpoint, this regression test:

1. locates the corresponding entry in the locked batch request plan;
2. reconstructs the canonical payload from the matching representation record;
3. reparses the saved visible response;
4. reapplies the locked JSON-schema and feature-grounding checks;
5. compares the newly parsed object with the object stored in the checkpoint; and
6. confirms that the newly calculated validity indicators agree with the previously recorded indicators.

All ten checkpoints must reproduce their original valid status. Any disagreement indicates that the batch validation implementation is not equivalent to the validated reliability-stage implementation, and formal inference must not begin.

This regression test reads only saved experimental inputs and responses. It does not load ground-truth labels and does not make a network request.


In [11]:
def describe_container_structure(container) -> dict:
    """
    Describe a container without printing experimental feature values.

    The helper supports pandas DataFrames, lists of records, and dictionaries.
    """
    description = {
        "container_type": type(container).__name__,
    }

    if isinstance(container, pd.DataFrame):
        description["row_count"] = len(container)
        description["columns"] = sorted(
            str(column) for column in container.columns
        )
        description["column_types"] = {
            str(column): str(dtype)
            for column, dtype in container.dtypes.items()
        }

    elif isinstance(container, list):
        description["record_count"] = len(container)

        if container:
            first_record = container[0]
            description["first_record_type"] = type(
                first_record
            ).__name__

            if isinstance(first_record, dict):
                description["first_record_keys"] = sorted(
                    first_record.keys()
                )
                description["first_record_value_types"] = {
                    key: type(value).__name__
                    for key, value in first_record.items()
                }

    elif isinstance(container, dict):
        description["keys"] = sorted(container.keys())
        description["value_types"] = {
            key: type(value).__name__
            for key, value in container.items()
        }

    return description


diagnostic_checkpoint = json.loads(
    batch_checkpoint_path(
        record_index=0,
        condition="structured",
    ).read_text(encoding="utf-8")
)

representation_structure_diagnostic = {
    "structured_records": describe_container_structure(
        structured_records
    ),
    "text_records": describe_container_structure(
        text_records
    ),
    "equivalence_validation": describe_container_structure(
        equivalence_validation
    ),
    "batch_request_plan": describe_container_structure(
        batch_request_plan
    ),
    "checkpoint": describe_container_structure(
        diagnostic_checkpoint
    ),
    "ground_truth_loaded": False,
    "network_request_made": False,
}

for object_name, structure in representation_structure_diagnostic.items():
    print(f"\n--- {object_name} ---")
    print(json.dumps(structure, indent=2, ensure_ascii=False))


--- structured_records ---
{
  "container_type": "list",
  "record_count": 200,
  "first_record_type": "dict",
  "first_record_keys": [
    "canonical_payload_sha256",
    "feature_set_id",
    "model_input",
    "sample_id"
  ],
  "first_record_value_types": {
    "sample_id": "str",
    "feature_set_id": "str",
    "canonical_payload_sha256": "str",
    "model_input": "str"
  }
}

--- text_records ---
{
  "container_type": "list",
  "record_count": 200,
  "first_record_type": "dict",
  "first_record_keys": [
    "canonical_payload_sha256",
    "feature_set_id",
    "model_input",
    "sample_id"
  ],
  "first_record_value_types": {
    "sample_id": "str",
    "feature_set_id": "str",
    "canonical_payload_sha256": "str",
    "model_input": "str"
  }
}

--- equivalence_validation ---
{
  "container_type": "DataFrame",
  "row_count": 200,
  "columns": [
    "canonical_feature_count",
    "sample_id",
    "sample_id_inside_structured_input",
    "sample_id_inside_text_input",
    "str

In [12]:
def describe_json_shape(value, depth: int = 0, maximum_depth: int = 4):
    """
    Describe JSON structure recursively without exposing scalar values.
    """
    if depth >= maximum_depth:
        return {"type": type(value).__name__}

    if isinstance(value, dict):
        return {
            "type": "dict",
            "keys": sorted(value.keys()),
            "children": {
                key: describe_json_shape(
                    child_value,
                    depth=depth + 1,
                    maximum_depth=maximum_depth,
                )
                for key, child_value in value.items()
            },
        }

    if isinstance(value, list):
        description = {
            "type": "list",
            "length": len(value),
        }

        if value:
            description["first_item"] = describe_json_shape(
                value[0],
                depth=depth + 1,
                maximum_depth=maximum_depth,
            )

        return description

    return {"type": type(value).__name__}


structured_model_input_text = structured_records[0]["model_input"]

try:
    structured_model_input_object = json.loads(
        structured_model_input_text
    )
    structured_model_input_is_json = True
    structured_model_input_shape = describe_json_shape(
        structured_model_input_object
    )
except json.JSONDecodeError:
    structured_model_input_is_json = False
    structured_model_input_shape = {
        "type": "str",
        "character_count": len(structured_model_input_text),
    }


diagnostic_checkpoint = json.loads(
    batch_checkpoint_path(
        record_index=0,
        condition="structured",
    ).read_text(encoding="utf-8")
)

diagnostic_visible_response = diagnostic_checkpoint["visible_response"]

try:
    diagnostic_response_object = json.loads(
        diagnostic_visible_response
    )
    visible_response_is_json = True
    visible_response_shape = describe_json_shape(
        diagnostic_response_object
    )
except json.JSONDecodeError:
    visible_response_is_json = False
    visible_response_shape = {
        "type": "str",
        "character_count": len(diagnostic_visible_response),
    }


print("--- structured model_input ---")
print(
    json.dumps(
        {
            "is_json": structured_model_input_is_json,
            "shape": structured_model_input_shape,
        },
        indent=2,
        ensure_ascii=False,
    )
)

print("\n--- saved visible_response ---")
print(
    json.dumps(
        {
            "is_json": visible_response_is_json,
            "shape": visible_response_shape,
        },
        indent=2,
        ensure_ascii=False,
    )
)

print("\n--- safety state ---")
print(
    json.dumps(
        {
            "ground_truth_loaded": False,
            "network_request_made": False,
        },
        indent=2,
    )
)

--- structured model_input ---
{
  "is_json": true,
  "shape": {
    "type": "dict",
    "keys": [
      "features",
      "record_type"
    ],
    "children": {
      "record_type": {
        "type": "str"
      },
      "features": {
        "type": "list",
        "length": 46,
        "first_item": {
          "type": "dict",
          "keys": [
            "name",
            "value"
          ],
          "children": {
            "name": {
              "type": "str"
            },
            "value": {
              "type": "int"
            }
          }
        }
      }
    }
  }
}

--- saved visible_response ---
{
  "is_json": true,
  "shape": {
    "type": "dict",
    "keys": [
      "cited_features",
      "predicted_label"
    ],
    "children": {
      "predicted_label": {
        "type": "str"
      },
      "cited_features": {
        "type": "list",
        "length": 5,
        "first_item": {
          "type": "dict",
          "keys": [
            "feature_name",

In [15]:
from jsonschema import Draft202012Validator


def find_response_object_schema(schema_document: dict) -> dict:
    """
    Locate the actual model-response schema inside a possibly wrapped
    schema document.

    The required response schema is identified structurally: its properties
    must contain both predicted_label and cited_features. This avoids
    depending on an assumed wrapper-field name.
    """
    matching_schemas = []

    def visit(value):
        if isinstance(value, dict):
            properties = value.get("properties")

            if (
                isinstance(properties, dict)
                and "predicted_label" in properties
                and "cited_features" in properties
            ):
                matching_schemas.append(value)

            for nested_value in value.values():
                visit(nested_value)

        elif isinstance(value, list):
            for nested_value in value:
                visit(nested_value)

    visit(schema_document)

    if len(matching_schemas) != 1:
        raise ValueError(
            "Expected exactly one response-object schema containing "
            "'predicted_label' and 'cited_features', but found "
            f"{len(matching_schemas)}."
        )

    return matching_schemas[0]


# Extract the real response schema from the locked schema-document wrapper.
RESPONSE_OBJECT_SCHEMA = find_response_object_schema(
    llm_output_schema
)

# Compile the actual response schema once. All formal responses will use
# this same validator.
Draft202012Validator.check_schema(RESPONSE_OBJECT_SCHEMA)
LLM_OUTPUT_VALIDATOR = Draft202012Validator(
    RESPONSE_OBJECT_SCHEMA
)

PREDICTED_LABEL_SCHEMA = RESPONSE_OBJECT_SCHEMA[
    "properties"
]["predicted_label"]

CITED_FEATURES_SCHEMA = RESPONSE_OBJECT_SCHEMA[
    "properties"
]["cited_features"]

ALLOWED_PREDICTED_LABELS = set(
    PREDICTED_LABEL_SCHEMA.get(
        "enum",
        ["Benign", "DoS"],
    )
)

MINIMUM_CITATION_COUNT = int(
    CITED_FEATURES_SCHEMA.get("minItems", 5)
)

MAXIMUM_CITATION_COUNT = int(
    CITED_FEATURES_SCHEMA.get("maxItems", 5)
)


def parse_opencode_event_stream(stdout_text: str) -> list[dict]:
    """
    Parse OpenCode's newline-delimited JSON event stream.

    Blank lines are ignored. Every other line must contain one JSON object.
    """
    events = []

    for line_number, raw_line in enumerate(
        stdout_text.splitlines(),
        start=1,
    ):
        stripped_line = raw_line.strip()

        if not stripped_line:
            continue

        try:
            event = json.loads(stripped_line)
        except json.JSONDecodeError as exc:
            raise ValueError(
                "OpenCode stdout contained a non-JSON event on "
                f"line {line_number}: {exc}"
            ) from exc

        if not isinstance(event, dict):
            raise ValueError(
                f"OpenCode event on line {line_number} is not an object."
            )

        events.append(event)

    if not events:
        raise ValueError("OpenCode returned no parseable JSON events.")

    return events


def extract_visible_response_from_events(events: list[dict]) -> str:
    """
    Reconstruct the visible assistant response from OpenCode text events.
    """
    visible_text_parts = []

    for event in events:
        if event.get("type") != "text":
            continue

        part = event.get("part", {})
        text_value = part.get("text")

        if isinstance(text_value, str) and text_value:
            visible_text_parts.append(text_value)

    visible_response = "".join(visible_text_parts).strip()

    if not visible_response:
        raise ValueError(
            "No non-empty visible response was found in the event stream."
        )

    return visible_response


def parse_visible_json_object(visible_response: str) -> dict:
    """
    Parse the visible response as exactly one bare JSON object.

    Markdown fences and additional explanatory prose are not accepted.
    """
    try:
        parsed_response = json.loads(visible_response)
    except json.JSONDecodeError as exc:
        raise ValueError(
            f"The visible response is not valid JSON: {exc}"
        ) from exc

    if not isinstance(parsed_response, dict):
        raise ValueError(
            "The visible response must parse to one JSON object."
        )

    return parsed_response


def validate_schema_structure(
    parsed_response: dict,
) -> tuple[bool, list[str]]:
    """Validate one response against the locked output schema."""
    schema_errors = sorted(
        LLM_OUTPUT_VALIDATOR.iter_errors(parsed_response),
        key=lambda error: list(error.absolute_path),
    )

    formatted_errors = []

    for error in schema_errors:
        error_path = ".".join(
            str(item) for item in error.absolute_path
        )
        location = error_path if error_path else "<root>"
        formatted_errors.append(f"{location}: {error.message}")

    return len(formatted_errors) == 0, formatted_errors


def extract_grounding_reference(
    structured_model_input: str,
) -> dict:
    """
    Recover the feature-name-to-value mapping from a structured model input.

    The structured and deterministic-text representations were previously
    verified to encode the same canonical payload. Therefore, this mapping
    is the common grounding reference for both conditions of the same
    record.
    """
    parsed_input = json.loads(structured_model_input)

    if not isinstance(parsed_input, dict):
        raise ValueError(
            "Structured model_input must parse to a JSON object."
        )

    features = parsed_input.get("features")

    if not isinstance(features, list):
        raise ValueError(
            "Structured model_input must contain a features list."
        )

    grounding_reference = {}

    for feature_position, feature in enumerate(features):
        if not isinstance(feature, dict):
            raise ValueError(
                f"Feature at position {feature_position} is not an object."
            )

        feature_name = feature.get("name")

        if not isinstance(feature_name, str) or not feature_name:
            raise ValueError(
                f"Feature at position {feature_position} has no valid name."
            )

        if "value" not in feature:
            raise ValueError(
                f"Feature {feature_name!r} has no value."
            )

        if feature_name in grounding_reference:
            raise ValueError(
                f"Duplicate feature name in structured input: {feature_name}"
            )

        grounding_reference[feature_name] = feature["value"]

    if len(grounding_reference) != 46:
        raise ValueError(
            "The primary grounding reference must contain 46 distinct "
            f"features, but found {len(grounding_reference)}."
        )

    return grounding_reference


def expected_observed_value_text(value) -> str:
    """
    Convert a supplied feature value to the exact text expected in a citation.

    Network-flow features in the locked primary representation are numeric.
    Python's numeric string representation matches the deterministic scalar
    representation used by the validated prompt contract.
    """
    return str(value)


def validate_feature_grounding(
    parsed_response: dict,
    grounding_reference: dict,
) -> dict:
    """
    Reproduce the detailed grounding checks stored in the 06 checkpoints.
    """
    predicted_label = parsed_response.get("predicted_label")
    cited_features = parsed_response.get("cited_features")

    predicted_label_valid = (
        isinstance(predicted_label, str)
        and predicted_label in ALLOWED_PREDICTED_LABELS
    )

    citation_count_valid = (
        isinstance(cited_features, list)
        and MINIMUM_CITATION_COUNT
        <= len(cited_features)
        <= MAXIMUM_CITATION_COUNT
    )

    cited_feature_names = []
    unsupported_feature_names = []
    observed_value_mismatches = []

    if isinstance(cited_features, list):
        for citation_position, citation in enumerate(cited_features):
            if not isinstance(citation, dict):
                unsupported_feature_names.append(
                    f"<malformed citation {citation_position}>"
                )
                continue

            feature_name = citation.get("feature_name")
            observed_value = citation.get("observed_value")

            if not isinstance(feature_name, str):
                unsupported_feature_names.append(
                    f"<missing feature name {citation_position}>"
                )
                continue

            cited_feature_names.append(feature_name)

            if feature_name not in grounding_reference:
                unsupported_feature_names.append(feature_name)
                continue

            expected_text = expected_observed_value_text(
                grounding_reference[feature_name]
            )

            if (
                not isinstance(observed_value, str)
                or observed_value.strip() != expected_text
            ):
                observed_value_mismatches.append(
                    {
                        "feature_name": feature_name,
                        "expected": expected_text,
                        "observed": observed_value,
                    }
                )

    supported_feature_names_valid = (
        citation_count_valid
        and len(unsupported_feature_names) == 0
        and len(cited_feature_names) == len(cited_features)
    )

    distinct_feature_names_valid = (
        citation_count_valid
        and len(cited_feature_names) == len(set(cited_feature_names))
    )

    observed_values_match = (
        citation_count_valid
        and supported_feature_names_valid
        and len(observed_value_mismatches) == 0
    )

    grounding_valid = all(
        [
            citation_count_valid,
            supported_feature_names_valid,
            distinct_feature_names_valid,
            observed_values_match,
        ]
    )

    return {
        "predicted_label_valid": predicted_label_valid,
        "citation_count_valid": citation_count_valid,
        "supported_feature_names_valid": (
            supported_feature_names_valid
        ),
        "distinct_feature_names_valid": (
            distinct_feature_names_valid
        ),
        "observed_values_match": observed_values_match,
        "grounding_valid": grounding_valid,
        "cited_feature_names": cited_feature_names,
        "unsupported_feature_names": sorted(
            set(unsupported_feature_names)
        ),
        "observed_value_mismatches": observed_value_mismatches,
    }


def validate_model_response(
    visible_response: str,
    grounding_reference: dict,
) -> dict:
    """
    Apply parsing, schema, label, citation, and grounding validation.

    Failed responses are returned as structured evidence rather than being
    discarded.
    """
    result = {
        "visible_json_parsed": False,
        "schema_structure_valid": False,
        "predicted_label_valid": False,
        "citation_count_valid": False,
        "supported_feature_names_valid": False,
        "distinct_feature_names_valid": False,
        "observed_values_match": False,
        "grounding_valid": False,
        "overall_response_valid": False,
        "parsed_response": None,
        "schema_validation_errors": [],
        "grounding_details": None,
        "error_type": None,
        "error_message": None,
    }

    try:
        parsed_response = parse_visible_json_object(visible_response)
        result["visible_json_parsed"] = True
        result["parsed_response"] = parsed_response

        schema_valid, schema_errors = validate_schema_structure(
            parsed_response
        )
        result["schema_structure_valid"] = schema_valid
        result["schema_validation_errors"] = schema_errors

        if schema_valid:
            grounding_details = validate_feature_grounding(
                parsed_response=parsed_response,
                grounding_reference=grounding_reference,
            )

            for field_name in [
                "predicted_label_valid",
                "citation_count_valid",
                "supported_feature_names_valid",
                "distinct_feature_names_valid",
                "observed_values_match",
                "grounding_valid",
            ]:
                result[field_name] = grounding_details[field_name]

            result["grounding_details"] = grounding_details

        result["overall_response_valid"] = all(
            [
                result["visible_json_parsed"],
                result["schema_structure_valid"],
                result["predicted_label_valid"],
                result["grounding_valid"],
            ]
        )

    except Exception as exc:
        result["error_type"] = type(exc).__name__
        result["error_message"] = str(exc)

    return result


validation_helper_summary = pd.Series(
    {
        "locked_schema_compiled": True,
        "allowed_predicted_labels": sorted(
            ALLOWED_PREDICTED_LABELS
        ),
        "minimum_citation_count": MINIMUM_CITATION_COUNT,
        "maximum_citation_count": MAXIMUM_CITATION_COUNT,
        "event_stream_parser_defined": True,
        "grounding_reference_extractor_defined": True,
        "detailed_grounding_validator_defined": True,
        "ground_truth_required": False,
        "ground_truth_loaded": False,
        "network_request_made": False,
    },
    name="value",
)

validation_helper_summary

locked_schema_compiled                            True
allowed_predicted_labels                 [Benign, DoS]
minimum_citation_count                               5
maximum_citation_count                               5
event_stream_parser_defined                       True
grounding_reference_extractor_defined             True
detailed_grounding_validator_defined              True
ground_truth_required                            False
ground_truth_loaded                              False
network_request_made                             False
Name: value, dtype: object

In [16]:
REGRESSION_VALIDATION_FIELDS = [
    "visible_json_parsed",
    "schema_structure_valid",
    "predicted_label_valid",
    "citation_count_valid",
    "supported_feature_names_valid",
    "distinct_feature_names_valid",
    "observed_values_match",
    "grounding_valid",
    "overall_response_valid",
]


# Select only the ten reliability checkpoints already admitted into the
# formal batch request plan.
reusable_plan_entries = [
    plan_entry
    for plan_entry in batch_request_plan
    if plan_entry["reusable_checkpoint_available"]
]

assert len(reusable_plan_entries) == 10

assert {
    plan_entry["record_index"]
    for plan_entry in reusable_plan_entries
} == expected_reliability_indices

assert {
    plan_entry["condition"]
    for plan_entry in reusable_plan_entries
} == expected_conditions


reusable_plan_by_key = {
    (
        plan_entry["record_index"],
        plan_entry["condition"],
    ): plan_entry
    for plan_entry in reusable_plan_entries
}

assert len(reusable_plan_by_key) == 10


regression_rows = []

for request_key, plan_entry in reusable_plan_by_key.items():
    record_index, condition = request_key

    formal_checkpoint_path = batch_checkpoint_path(
        record_index=record_index,
        condition=condition,
    )

    reliability_checkpoint_path = (
        RELIABILITY_CHECKPOINT_DIR
        / f"record_{record_index:04d}__{condition}.json"
    )

    if not formal_checkpoint_path.is_file():
        raise FileNotFoundError(
            "Missing seeded formal checkpoint: "
            f"{formal_checkpoint_path}"
        )

    if not reliability_checkpoint_path.is_file():
        raise FileNotFoundError(
            "Missing source reliability checkpoint: "
            f"{reliability_checkpoint_path}"
        )

    formal_checkpoint = json.loads(
        formal_checkpoint_path.read_text(
            encoding="utf-8"
        )
    )

    reliability_checkpoint = json.loads(
        reliability_checkpoint_path.read_text(
            encoding="utf-8"
        )
    )

    # The formal checkpoint must remain a byte-for-byte copy of the
    # original validated reliability checkpoint.
    formal_sha256 = sha256_file(
        formal_checkpoint_path
    )

    reliability_sha256 = sha256_file(
        reliability_checkpoint_path
    )

    exact_source_byte_hash_match = (
        formal_sha256 == reliability_sha256
    )

    assert exact_source_byte_hash_match
    assert formal_checkpoint == reliability_checkpoint

    structured_record = structured_records[
        record_index
    ]

    text_record = text_records[
        record_index
    ]

    # Reconfirm paired-input identity.
    assert (
        structured_record["sample_id"]
        == text_record["sample_id"]
    )

    assert (
        structured_record[
            "canonical_payload_sha256"
        ]
        == text_record[
            "canonical_payload_sha256"
        ]
    )

    # Reconfirm checkpoint identity and hashes against the current
    # locked batch request plan.
    assert (
        formal_checkpoint["record_index"]
        == record_index
    )

    assert (
        formal_checkpoint["sample_id"]
        == plan_entry["sample_id"]
        == structured_record["sample_id"]
    )

    assert (
        formal_checkpoint["condition"]
        == condition
    )

    assert (
        formal_checkpoint[
            "canonical_payload_sha256"
        ]
        == plan_entry[
            "canonical_payload_sha256"
        ]
        == structured_record[
            "canonical_payload_sha256"
        ]
    )

    assert (
        formal_checkpoint["prompt_sha256"]
        == plan_entry["prompt_sha256"]
    )

    assert isinstance(
        formal_checkpoint[
            "execution_contract_sha256"
        ],
        str,
    )

    assert len(
        formal_checkpoint[
            "execution_contract_sha256"
        ]
    ) == 64

    # Reconfirm the locked backend and safety configuration.
    expected_configuration = {
        "feature_set_id": "primary_46",
        "backend": "opencode",
        "opencode_version": OPENCODE_VERSION,
        "requested_model": OPENCODE_MODEL_ROUTE,
        "prompt_template_version": (
            OPENCODE_PROMPT_TEMPLATE_VERSION
        ),
        "subprocess_timeout_seconds": (
            OPENCODE_REQUEST_TIMEOUT_SECONDS
        ),
        "pure_mode_used": True,
        "empty_temporary_directory_used": True,
        "files_attached": False,
        "auto_approval_used": False,
        "maximum_model_output_tokens_supplied": False,
        "sample_id_sent_to_model": False,
        "ground_truth_loaded": False,
    }

    for field_name, expected_value in (
        expected_configuration.items()
    ):
        assert (
            formal_checkpoint[field_name]
            == expected_value
        ), (
            f"Configuration mismatch for "
            f"{request_key}, field={field_name!r}"
        )

    # Reconfirm successful stored execution state.
    assert (
        formal_checkpoint["request_completed"]
        is True
    )

    assert formal_checkpoint["return_code"] == 0

    assert (
        formal_checkpoint["budget_exhausted"]
        is False
    )

    assert formal_checkpoint["error_type"] is None
    assert formal_checkpoint["error_message"] is None
    assert formal_checkpoint["validation_error"] is None

    assert (
        formal_checkpoint[
            "visible_response_character_count"
        ]
        == len(
            formal_checkpoint["visible_response"]
        )
    )

    # Use the structured representation as the common grounding
    # reference for both equivalent conditions of this record.
    grounding_reference = (
        extract_grounding_reference(
            structured_record["model_input"]
        )
    )

    revalidation = validate_model_response(
        visible_response=(
            formal_checkpoint["visible_response"]
        ),
        grounding_reference=grounding_reference,
    )

    assert (
        revalidation["schema_validation_errors"]
        == []
    )

    assert revalidation["error_type"] is None
    assert revalidation["error_message"] is None

    assert (
        revalidation["parsed_response"]
        == parse_visible_json_object(
            formal_checkpoint[
                "visible_response"
            ]
        )
    )

    # Every newly calculated validation result must pass and must
    # reproduce the corresponding stored result.
    for validation_field in (
        REGRESSION_VALIDATION_FIELDS
    ):
        assert (
            revalidation[validation_field]
            is True
        ), (
            f"Revalidation failed for "
            f"{request_key}, "
            f"field={validation_field!r}"
        )

        if validation_field in formal_checkpoint:
            assert (
                formal_checkpoint[
                    validation_field
                ]
                == revalidation[
                    validation_field
                ]
            )

    grounding_details = revalidation[
        "grounding_details"
    ]

    assert isinstance(
        grounding_details,
        dict,
    )

    assert len(
        grounding_details[
            "cited_feature_names"
        ]
    ) == 5

    assert (
        grounding_details[
            "unsupported_feature_names"
        ]
        == []
    )

    assert (
        grounding_details[
            "observed_value_mismatches"
        ]
        == []
    )

    regression_rows.append(
        {
            "record_index": record_index,
            "sample_id": plan_entry[
                "sample_id"
            ],
            "condition": condition,
            "exact_source_byte_hash_match": (
                exact_source_byte_hash_match
            ),
            **{
                validation_field: (
                    revalidation[
                        validation_field
                    ]
                )
                for validation_field
                in REGRESSION_VALIDATION_FIELDS
            },
            "schema_validation_error_count": len(
                revalidation[
                    "schema_validation_errors"
                ]
            ),
            "unsupported_feature_name_count": len(
                grounding_details[
                    "unsupported_feature_names"
                ]
            ),
            "observed_value_mismatch_count": len(
                grounding_details[
                    "observed_value_mismatches"
                ]
            ),
            "ground_truth_loaded": False,
            "network_request_made": False,
        }
    )


regression_revalidation_table = (
    pd.DataFrame(regression_rows)
    .sort_values(
        ["record_index", "condition"]
    )
    .reset_index(drop=True)
)


assert len(regression_revalidation_table) == 10

assert regression_revalidation_table[
    "exact_source_byte_hash_match"
].all()

assert regression_revalidation_table[
    REGRESSION_VALIDATION_FIELDS
].to_numpy().all()

assert (
    regression_revalidation_table[
        "schema_validation_error_count"
    ]
    == 0
).all()

assert (
    regression_revalidation_table[
        "unsupported_feature_name_count"
    ]
    == 0
).all()

assert (
    regression_revalidation_table[
        "observed_value_mismatch_count"
    ]
    == 0
).all()

assert not regression_revalidation_table[
    "ground_truth_loaded"
].any()

assert not regression_revalidation_table[
    "network_request_made"
].any()


display(regression_revalidation_table)


regression_gate_summary = pd.Series(
    {
        "seeded_checkpoint_count": len(
            regression_revalidation_table
        ),
        "exact_source_byte_hash_matches": int(
            regression_revalidation_table[
                "exact_source_byte_hash_match"
            ].sum()
        ),
        "responses_reparsed": int(
            regression_revalidation_table[
                "visible_json_parsed"
            ].sum()
        ),
        "schemas_valid": int(
            regression_revalidation_table[
                "schema_structure_valid"
            ].sum()
        ),
        "labels_valid": int(
            regression_revalidation_table[
                "predicted_label_valid"
            ].sum()
        ),
        "citation_counts_valid": int(
            regression_revalidation_table[
                "citation_count_valid"
            ].sum()
        ),
        "distinct_feature_names_valid": int(
            regression_revalidation_table[
                "distinct_feature_names_valid"
            ].sum()
        ),
        "supported_feature_names_valid": int(
            regression_revalidation_table[
                "supported_feature_names_valid"
            ].sum()
        ),
        "observed_values_match": int(
            regression_revalidation_table[
                "observed_values_match"
            ].sum()
        ),
        "grounding_valid": int(
            regression_revalidation_table[
                "grounding_valid"
            ].sum()
        ),
        "overall_responses_valid": int(
            regression_revalidation_table[
                "overall_response_valid"
            ].sum()
        ),
        "formal_inference_gate_passed": True,
        "ground_truth_loaded": False,
        "network_request_made": False,
    },
    name="value",
)

regression_gate_summary

,record_index,sample_id,condition,exact_source_byte_hash_match,visible_json_parsed,schema_structure_valid,predicted_label_valid,citation_count_valid,supported_feature_names_valid,distinct_feature_names_valid,observed_values_match,grounding_valid,overall_response_valid,schema_validation_error_count,unsupported_feature_name_count,observed_value_mismatch_count,ground_truth_loaded,network_request_made
0,0,pilot_001,deterministic_text,True,True,True,True,True,True,True,True,True,True,0,0,0,False,False
1,0,pilot_001,structured,True,True,True,True,True,True,True,True,True,True,0,0,0,False,False
2,50,pilot_051,deterministic_text,True,True,True,True,True,True,True,True,True,True,0,0,0,False,False
3,50,pilot_051,structured,True,True,True,True,True,True,True,True,True,True,0,0,0,False,False
4,100,pilot_101,deterministic_text,True,True,True,True,True,True,True,True,True,True,0,0,0,False,False
5,100,pilot_101,structured,True,True,True,True,True,True,True,True,True,True,0,0,0,False,False
6,150,pilot_151,deterministic_text,True,True,True,True,True,True,True,True,True,True,0,0,0,False,False
7,150,pilot_151,structured,True,True,True,True,True,True,True,True,True,True,0,0,0,False,False
8,199,pilot_200,deterministic_text,True,True,True,True,True,True,True,True,True,True,0,0,0,False,False
9,199,pilot_200,structured,True,True,True,True,True,True,True,True,True,True,0,0,0,False,False


seeded_checkpoint_count              10
exact_source_byte_hash_matches       10
responses_reparsed                   10
schemas_valid                        10
labels_valid                         10
citation_counts_valid                10
distinct_feature_names_valid         10
supported_feature_names_valid        10
observed_values_match                10
grounding_valid                      10
overall_responses_valid              10
formal_inference_gate_passed       True
ground_truth_loaded               False
network_request_made              False
Name: value, dtype: object

In [18]:
import os


def serialise_checkpoint(
    checkpoint: dict,
) -> bytes:
    """
    Serialise one checkpoint deterministically as UTF-8 JSON.

    A trailing newline gives all newly generated checkpoints the same
    stable text-file convention.
    """
    if not isinstance(checkpoint, dict):
        raise TypeError(
            "checkpoint must be a dictionary."
        )

    serialised_text = json.dumps(
        checkpoint,
        ensure_ascii=False,
        sort_keys=True,
        indent=2,
    )

    return (
        serialised_text + "\n"
    ).encode("utf-8")


def write_checkpoint_atomically(
    destination_path: Path,
    checkpoint: dict,
) -> dict:
    """
    Atomically create one checkpoint without overwriting evidence.

    If the destination already exists:
    - identical bytes are verified and reused;
    - different bytes cause an immediate error.
    """
    destination_path = Path(
        destination_path
    )

    checkpoint_bytes = serialise_checkpoint(
        checkpoint
    )

    expected_sha256 = hashlib.sha256(
        checkpoint_bytes
    ).hexdigest()

    destination_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if destination_path.exists():
        existing_bytes = (
            destination_path.read_bytes()
        )

        existing_sha256 = hashlib.sha256(
            existing_bytes
        ).hexdigest()

        if existing_bytes != checkpoint_bytes:
            raise RuntimeError(
                "Refusing to overwrite divergent "
                f"checkpoint evidence: {destination_path}"
            )

        return {
            "checkpoint_action": (
                "verified_existing"
            ),
            "checkpoint_sha256": (
                existing_sha256
            ),
            "checkpoint_path": str(
                destination_path
            ),
        }

    temporary_path = None

    try:
        with tempfile.NamedTemporaryFile(
            mode="wb",
            dir=destination_path.parent,
            prefix=(
                destination_path.name
                + "."
            ),
            suffix=".tmp",
            delete=False,
        ) as temporary_file:
            temporary_path = Path(
                temporary_file.name
            )

            temporary_file.write(
                checkpoint_bytes
            )

            temporary_file.flush()

            os.fsync(
                temporary_file.fileno()
            )

        temporary_bytes = (
            temporary_path.read_bytes()
        )

        temporary_sha256 = hashlib.sha256(
            temporary_bytes
        ).hexdigest()

        if temporary_sha256 != expected_sha256:
            raise RuntimeError(
                "Temporary checkpoint failed "
                "SHA-256 verification."
            )

        try:
            # The hard-link operation is atomic and refuses to replace
            # an existing destination.
            os.link(
                temporary_path,
                destination_path,
            )

            checkpoint_action = "created"

        except FileExistsError:
            existing_bytes = (
                destination_path.read_bytes()
            )

            if existing_bytes != checkpoint_bytes:
                raise RuntimeError(
                    "A divergent checkpoint appeared "
                    "during the atomic write: "
                    f"{destination_path}"
                )

            checkpoint_action = (
                "verified_existing"
            )

    finally:
        if (
            temporary_path is not None
            and temporary_path.exists()
        ):
            temporary_path.unlink()

    final_bytes = (
        destination_path.read_bytes()
    )

    final_sha256 = hashlib.sha256(
        final_bytes
    ).hexdigest()

    if final_bytes != checkpoint_bytes:
        raise RuntimeError(
            "Final checkpoint bytes do not match "
            f"the intended evidence: {destination_path}"
        )

    if final_sha256 != expected_sha256:
        raise RuntimeError(
            "Final checkpoint failed SHA-256 "
            f"verification: {destination_path}"
        )

    return {
        "checkpoint_action": (
            checkpoint_action
        ),
        "checkpoint_sha256": (
            final_sha256
        ),
        "checkpoint_path": str(
            destination_path
        ),
    }


# Test only inside a disposable directory.
# The formal checkpoint directory is not touched.
with tempfile.TemporaryDirectory() as (
    writer_test_directory
):
    writer_test_path = (
        Path(writer_test_directory)
        / "writer_test_checkpoint.json"
    )

    writer_test_checkpoint = {
        "test_id": "atomic-writer-regression",
        "status": "complete",
    }

    first_write_result = (
        write_checkpoint_atomically(
            destination_path=(
                writer_test_path
            ),
            checkpoint=(
                writer_test_checkpoint
            ),
        )
    )

    second_write_result = (
        write_checkpoint_atomically(
            destination_path=(
                writer_test_path
            ),
            checkpoint=(
                writer_test_checkpoint
            ),
        )
    )

    divergent_overwrite_blocked = False

    try:
        write_checkpoint_atomically(
            destination_path=(
                writer_test_path
            ),
            checkpoint={
                "test_id": (
                    "atomic-writer-regression"
                ),
                "status": "divergent",
            },
        )

    except RuntimeError:
        divergent_overwrite_blocked = True

    assert (
        first_write_result[
            "checkpoint_action"
        ]
        == "created"
    )

    assert (
        second_write_result[
            "checkpoint_action"
        ]
        == "verified_existing"
    )

    assert divergent_overwrite_blocked

    assert json.loads(
        writer_test_path.read_text(
            encoding="utf-8"
        )
    ) == writer_test_checkpoint


atomic_writer_summary = pd.Series(
    {
        "writer_defined": True,
        "first_write_created": (
            first_write_result[
                "checkpoint_action"
            ]
            == "created"
        ),
        "identical_existing_file_verified": (
            second_write_result[
                "checkpoint_action"
            ]
            == "verified_existing"
        ),
        "divergent_overwrite_blocked": (
            divergent_overwrite_blocked
        ),
        "formal_checkpoint_directory_modified": False,
        "ground_truth_loaded": False,
        "network_request_made": False,
    },
    name="value",
)

atomic_writer_summary

writer_defined                           True
first_write_created                      True
identical_existing_file_verified         True
divergent_overwrite_blocked              True
formal_checkpoint_directory_modified    False
ground_truth_loaded                     False
network_request_made                    False
Name: value, dtype: bool

In [24]:
OPENCODE_BUDGET_EXHAUSTED_MARKERS = [
    "insufficient_quota",
    "quota exceeded",
    "credit balance",
    "out of credits",
    "usage limit",
    "payment required",
]

FORMAL_ALLOWED_CONDITIONS = {
    "structured",
    "deterministic_text",
}


def build_opencode_execution_contract(
    *,
    prompt_sha256: str,
) -> dict:
    """
    Reproduce the provider-specific execution contract validated in
    Notebook 06.

    The contract is request-specific because it includes prompt_sha256.
    """
    if (
        not isinstance(prompt_sha256, str)
        or len(prompt_sha256) != 64
    ):
        raise ValueError(
            "prompt_sha256 must be a "
            "64-character SHA-256 digest."
        )

    return {
        "backend": "opencode",
        "opencode_version": (
            OPENCODE_VERSION
        ),
        "executable": str(
            OPENCODE_EXECUTABLE
        ),
        "model": (
            OPENCODE_MODEL_ROUTE
        ),
        "prompt_template_version": (
            OPENCODE_PROMPT_TEMPLATE_VERSION
        ),
        "prompt_sha256": (
            prompt_sha256
        ),
        "pure": True,
        "empty_temporary_directory": True,
        "files_attached": False,
        "auto_approval": False,
        "format": "json",
        "timeout_seconds": (
            OPENCODE_REQUEST_TIMEOUT_SECONDS
        ),
        "maximum_model_output_tokens": None,
    }


def summarise_opencode_research_events(
    events: list[dict],
) -> dict:
    """
    Extract operational metadata from an already strictly parsed
    OpenCode event stream.

    Raw events and hidden reasoning text are not retained.
    """
    event_types = [
        event.get("type")
        for event in events
    ]

    finish_reasons = []

    input_tokens = 0
    output_tokens = 0
    reasoning_tokens = 0
    total_tokens = 0
    cache_write_tokens = 0
    cache_read_tokens = 0
    reported_cost = 0.0
    step_finish_count = 0

    for event in events:
        if (
            event.get("type")
            != "step_finish"
        ):
            continue

        step_finish_count += 1

        part = event.get("part")

        if not isinstance(part, dict):
            continue

        reason = part.get("reason")

        if reason is not None:
            finish_reasons.append(
                str(reason)
            )

        tokens = part.get("tokens")

        if isinstance(tokens, dict):
            input_tokens += int(
                tokens.get("input") or 0
            )

            output_tokens += int(
                tokens.get("output") or 0
            )

            reasoning_tokens += int(
                tokens.get("reasoning") or 0
            )

            total_tokens += int(
                tokens.get("total") or 0
            )

            cache_tokens = tokens.get(
                "cache"
            )

            if isinstance(
                cache_tokens,
                dict,
            ):
                cache_write_tokens += int(
                    cache_tokens.get(
                        "write"
                    )
                    or 0
                )

                cache_read_tokens += int(
                    cache_tokens.get(
                        "read"
                    )
                    or 0
                )

        cost = part.get("cost")

        if isinstance(
            cost,
            (int, float),
        ):
            reported_cost += float(
                cost
            )

    return {
        "event_count": len(events),
        "event_types": event_types,
        "malformed_line_count": 0,
        "step_finish_count": (
            step_finish_count
        ),
        "finish_reasons": (
            finish_reasons
        ),
        "input_tokens": (
            input_tokens
        ),
        "output_tokens": (
            output_tokens
        ),
        "reasoning_tokens": (
            reasoning_tokens
        ),
        "total_tokens": (
            total_tokens
        ),
        "cache_write_tokens": (
            cache_write_tokens
        ),
        "cache_read_tokens": (
            cache_read_tokens
        ),
        "reported_cost": (
            reported_cost
        ),
    }


def coerce_subprocess_text(
    value,
) -> str:
    """
    Convert optional subprocess stdout/stderr evidence to bounded text.
    """
    if value is None:
        return ""

    if isinstance(value, bytes):
        return value.decode(
            "utf-8",
            errors="replace",
        )

    return str(value)


def summarise_failed_event_stream(
    stdout_text: str,
) -> dict:
    """
    Preserve structural event evidence after strict parsing fails.

    This deliberately does not extract or retain any visible or hidden
    event text.
    """
    parsed_events = []
    malformed_line_count = 0

    for raw_line in (
        stdout_text.splitlines()
    ):
        stripped_line = (
            raw_line.strip()
        )

        if not stripped_line:
            continue

        try:
            event = json.loads(
                stripped_line
            )
        except json.JSONDecodeError:
            malformed_line_count += 1
            continue

        if isinstance(event, dict):
            parsed_events.append(
                event
            )
        else:
            malformed_line_count += 1

    if parsed_events:
        summary = (
            summarise_opencode_research_events(
                parsed_events
            )
        )
    else:
        summary = {
            "event_count": 0,
            "event_types": [],
            "malformed_line_count": 0,
            "step_finish_count": 0,
            "finish_reasons": [],
            "input_tokens": None,
            "output_tokens": None,
            "reasoning_tokens": None,
            "total_tokens": None,
            "cache_write_tokens": None,
            "cache_read_tokens": None,
            "reported_cost": None,
        }

    summary[
        "malformed_line_count"
    ] = malformed_line_count

    return summary


def make_empty_response_validation() -> dict:
    """
    Return the validation state used when no complete visible response
    can be accepted.
    """
    return {
        "visible_json_parsed": False,
        "schema_structure_valid": False,
        "predicted_label_valid": False,
        "citation_count_valid": False,
        "supported_feature_names_valid": False,
        "distinct_feature_names_valid": False,
        "observed_values_match": False,
        "grounding_valid": False,
        "overall_response_valid": False,
        "schema_validation_errors": [],
        "grounding_details": None,
        "validation_error": None,
    }


def prepare_response_validation_for_checkpoint(
    validation_result: dict,
) -> dict:
    """
    Convert the corrected Notebook 07 validator result into checkpoint
    fields.

    parsed_response is not duplicated because the original visible JSON
    text is already retained in visible_response.
    """
    validation_error = None

    if not validation_result[
        "overall_response_valid"
    ]:
        if validation_result[
            "error_message"
        ]:
            validation_error = (
                f"{validation_result['error_type']}: "
                f"{validation_result['error_message']}"
            )[:500]

        elif validation_result[
            "schema_validation_errors"
        ]:
            validation_error = (
                "; ".join(
                    validation_result[
                        "schema_validation_errors"
                    ]
                )
            )[:500]

        else:
            validation_error = (
                "The response parsed but failed "
                "one or more label or grounding "
                "checks."
            )

    return {
        "visible_json_parsed": (
            validation_result[
                "visible_json_parsed"
            ]
        ),
        "schema_structure_valid": (
            validation_result[
                "schema_structure_valid"
            ]
        ),
        "predicted_label_valid": (
            validation_result[
                "predicted_label_valid"
            ]
        ),
        "citation_count_valid": (
            validation_result[
                "citation_count_valid"
            ]
        ),
        "supported_feature_names_valid": (
            validation_result[
                "supported_feature_names_valid"
            ]
        ),
        "distinct_feature_names_valid": (
            validation_result[
                "distinct_feature_names_valid"
            ]
        ),
        "observed_values_match": (
            validation_result[
                "observed_values_match"
            ]
        ),
        "grounding_valid": (
            validation_result[
                "grounding_valid"
            ]
        ),
        "overall_response_valid": (
            validation_result[
                "overall_response_valid"
            ]
        ),
        "schema_validation_errors": (
            validation_result[
                "schema_validation_errors"
            ]
        ),
        "grounding_details": (
            validation_result[
                "grounding_details"
            ]
        ),
        "validation_error": (
            validation_error
        ),
    }


def make_common_request_record(
    *,
    plan_entry: dict,
    execution_contract_sha256: str,
    request_started_at_utc: str,
    elapsed_seconds: float,
) -> dict:
    """
    Construct fields shared by successful and failed request attempts.
    """
    return {
        "record_index": (
            plan_entry[
                "record_index"
            ]
        ),
        "sample_id": (
            plan_entry["sample_id"]
        ),
        "condition": (
            plan_entry["condition"]
        ),
        "feature_set_id": (
            "primary_46"
        ),
        "backend": "opencode",
        "opencode_version": (
            OPENCODE_VERSION
        ),
        "requested_model": (
            OPENCODE_MODEL_ROUTE
        ),
        "canonical_payload_sha256": (
            plan_entry[
                "canonical_payload_sha256"
            ]
        ),
        "prompt_template_version": (
            OPENCODE_PROMPT_TEMPLATE_VERSION
        ),
        "prompt_sha256": (
            plan_entry[
                "prompt_sha256"
            ]
        ),
        "execution_contract_sha256": (
            execution_contract_sha256
        ),
        "request_started_at_utc": (
            request_started_at_utc
        ),
        "elapsed_seconds": (
            elapsed_seconds
        ),
        "pure_mode_used": True,
        "empty_temporary_directory_used": True,
        "auto_approval_used": False,
        "files_attached": False,
        "maximum_model_output_tokens_supplied": False,
        "subprocess_timeout_seconds": (
            OPENCODE_REQUEST_TIMEOUT_SECONDS
        ),
        "sample_id_sent_to_model": False,
        "ground_truth_loaded": False,
    }


def run_formal_opencode_request(
    *,
    plan_entry: dict,
    grounding_reference: dict,
) -> dict:
    """
    Execute exactly one formal OpenCode request.

    This function performs no automatic retry and does not write the
    result. The scheduler will persist each returned result immediately.
    """
    record_index = (
        plan_entry["record_index"]
    )

    sample_id = (
        plan_entry["sample_id"]
    )

    condition = (
        plan_entry["condition"]
    )

    prompt = (
        plan_entry["prompt"]
    )

    prompt_sha256 = (
        plan_entry["prompt_sha256"]
    )

    if condition not in (
        FORMAL_ALLOWED_CONDITIONS
    ):
        raise ValueError(
            "Unsupported representation "
            f"condition: {condition!r}."
        )

    if (
        not isinstance(prompt, str)
        or not prompt.strip()
    ):
        raise ValueError(
            "prompt must be a non-empty string."
        )

    if sample_id in prompt:
        raise ValueError(
            "The local sample identifier "
            "must not appear in the prompt."
        )

    expected_prompt_sha256 = (
        stable_json_sha256(
            {
                "template_version": (
                    OPENCODE_PROMPT_TEMPLATE_VERSION
                ),
                "prompt": prompt,
            }
        )
    )

    if (
        prompt_sha256
        != expected_prompt_sha256
    ):
        raise ValueError(
            "The supplied prompt hash does "
            "not match the prompt content."
        )

    execution_contract = (
        build_opencode_execution_contract(
            prompt_sha256=(
                prompt_sha256
            )
        )
    )

    execution_contract_sha256 = (
        stable_json_sha256(
            execution_contract
        )
    )

    request_started_at_utc = (
        datetime.now(
            timezone.utc
        ).isoformat()
    )

    request_start_time = (
        time.perf_counter()
    )

    request_was_started = False
    completed_process = None

    try:
        with tempfile.TemporaryDirectory(
            prefix=(
                "compsci742_opencode_research_"
            )
        ) as temporary_context_directory:
            temporary_context_path = Path(
                temporary_context_directory
            )

            if any(
                temporary_context_path.iterdir()
            ):
                raise RuntimeError(
                    "The temporary request "
                    "directory is not empty."
                )

            request_was_started = True

            completed_process = subprocess.run(
                [
                    str(
                        OPENCODE_EXECUTABLE
                    ),
                    "run",
                    prompt,
                    "--dir",
                    temporary_context_directory,
                    "--pure",
                    "--model",
                    OPENCODE_MODEL_ROUTE,
                    "--format",
                    "json",
                ],
                capture_output=True,
                text=True,
                timeout=(
                    OPENCODE_REQUEST_TIMEOUT_SECONDS
                ),
                check=False,
            )

        elapsed_seconds = (
            time.perf_counter()
            - request_start_time
        )

        events = (
            parse_opencode_event_stream(
                completed_process.stdout
            )
        )

        event_summary = (
            summarise_opencode_research_events(
                events
            )
        )

        visible_response = (
            extract_visible_response_from_events(
                events
            )
        )

        validation_result = (
            validate_model_response(
                visible_response=(
                    visible_response
                ),
                grounding_reference=(
                    grounding_reference
                ),
            )
        )

        checkpoint_validation = (
            prepare_response_validation_for_checkpoint(
                validation_result
            )
        )

        diagnostic_text = (
            completed_process.stdout
            + "\n"
            + completed_process.stderr
        ).lower()

        budget_exhausted = any(
            marker in diagnostic_text
            for marker in (
                OPENCODE_BUDGET_EXHAUSTED_MARKERS
            )
        )

        common_record = (
            make_common_request_record(
                plan_entry=plan_entry,
                execution_contract_sha256=(
                    execution_contract_sha256
                ),
                request_started_at_utc=(
                    request_started_at_utc
                ),
                elapsed_seconds=(
                    elapsed_seconds
                ),
            )
        )

        return {
            **common_record,
            "request_completed": True,
            "return_code": (
                completed_process.returncode
            ),
            **event_summary,
            "visible_response": (
                visible_response
            ),
            "visible_response_character_count": len(
                visible_response
            ),
            "budget_exhausted": (
                budget_exhausted
            ),
            "stderr_present": bool(
                completed_process
                .stderr
                .strip()
            ),
            "stderr_excerpt": (
                completed_process
                .stderr
                .strip()[:500]
            ),
            "research_data_transmitted": True,
            **checkpoint_validation,
            "error_type": None,
            "error_message": None,
        }

    except subprocess.TimeoutExpired as exc:
        elapsed_seconds = (
            time.perf_counter()
            - request_start_time
        )

        timeout_stdout = (
            coerce_subprocess_text(
                exc.stdout
            )
        )

        timeout_stderr = (
            coerce_subprocess_text(
                exc.stderr
            ).strip()
        )

        diagnostic_text = (
            timeout_stdout
            + "\n"
            + timeout_stderr
            + "\n"
            + str(exc)
        ).lower()

        budget_exhausted = any(
            marker in diagnostic_text
            for marker in (
                OPENCODE_BUDGET_EXHAUSTED_MARKERS
            )
        )

        common_record = (
            make_common_request_record(
                plan_entry=plan_entry,
                execution_contract_sha256=(
                    execution_contract_sha256
                ),
                request_started_at_utc=(
                    request_started_at_utc
                ),
                elapsed_seconds=(
                    elapsed_seconds
                ),
            )
        )

        return {
            **common_record,
            "request_completed": False,
            "return_code": None,
            "event_count": 0,
            "event_types": [],
            "malformed_line_count": 0,
            "step_finish_count": 0,
            "finish_reasons": [],
            "visible_response": "",
            "visible_response_character_count": 0,
            "input_tokens": None,
            "output_tokens": None,
            "reasoning_tokens": None,
            "total_tokens": None,
            "cache_write_tokens": None,
            "cache_read_tokens": None,
            "reported_cost": None,
            "budget_exhausted": (
                budget_exhausted
            ),
            "stderr_present": bool(
                timeout_stderr
            ),
            "stderr_excerpt": (
                timeout_stderr[:500]
            ),
            "research_data_transmitted": (
                request_was_started
            ),
            **make_empty_response_validation(),
            "error_type": (
                type(exc).__name__
            ),
            "error_message": (
                str(exc)[:500]
            ),
        }

    except Exception as exc:
        elapsed_seconds = (
            time.perf_counter()
            - request_start_time
        )

        stdout_text = ""
        stderr_text = ""

        if completed_process is not None:
            stdout_text = (
                completed_process.stdout
            )

            stderr_text = (
                completed_process
                .stderr
                .strip()
            )

        diagnostic_text = (
            stdout_text
            + "\n"
            + stderr_text
            + "\n"
            + str(exc)
        ).lower()

        budget_exhausted = any(
            marker in diagnostic_text
            for marker in (
                OPENCODE_BUDGET_EXHAUSTED_MARKERS
            )
        )

        failure_event_summary = (
            summarise_failed_event_stream(
                stdout_text
            )
        )

        common_record = (
            make_common_request_record(
                plan_entry=plan_entry,
                execution_contract_sha256=(
                    execution_contract_sha256
                ),
                request_started_at_utc=(
                    request_started_at_utc
                ),
                elapsed_seconds=(
                    elapsed_seconds
                ),
            )
        )

        return {
            **common_record,
            "request_completed": (
                completed_process
                is not None
            ),
            "return_code": (
                None
                if completed_process is None
                else completed_process.returncode
            ),
            **failure_event_summary,
            "visible_response": "",
            "visible_response_character_count": 0,
            "budget_exhausted": (
                budget_exhausted
            ),
            "stderr_present": bool(
                stderr_text
            ),
            "stderr_excerpt": (
                stderr_text[:500]
            ),
            "research_data_transmitted": (
                request_was_started
            ),
            **make_empty_response_validation(),
            "error_type": (
                type(exc).__name__
            ),
            "error_message": (
                str(exc)[:500]
            ),
        }


# Reproduce the request-specific execution-contract hashes recorded in
# all ten Notebook 06 checkpoints.
execution_contract_regression_rows = []

for plan_entry in (
    reusable_plan_entries
):
    checkpoint_path = (
        batch_checkpoint_path(
            record_index=(
                plan_entry[
                    "record_index"
                ]
            ),
            condition=(
                plan_entry[
                    "condition"
                ]
            ),
        )
    )

    checkpoint = json.loads(
        checkpoint_path.read_text(
            encoding="utf-8"
        )
    )

    reconstructed_contract = (
        build_opencode_execution_contract(
            prompt_sha256=(
                plan_entry[
                    "prompt_sha256"
                ]
            )
        )
    )

    reconstructed_sha256 = (
        stable_json_sha256(
            reconstructed_contract
        )
    )

    contract_hash_matches = (
        reconstructed_sha256
        == checkpoint[
            "execution_contract_sha256"
        ]
    )

    assert contract_hash_matches

    execution_contract_regression_rows.append(
        {
            "record_index": (
                plan_entry[
                    "record_index"
                ]
            ),
            "condition": (
                plan_entry[
                    "condition"
                ]
            ),
            "contract_hash_matches": (
                contract_hash_matches
            ),
        }
    )


execution_contract_regression_table = (
    pd.DataFrame(
        execution_contract_regression_rows
    )
    .sort_values(
        ["record_index", "condition"]
    )
    .reset_index(drop=True)
)

assert (
    len(
        execution_contract_regression_table
    )
    == 10
)

assert execution_contract_regression_table[
    "contract_hash_matches"
].all()


display(
    execution_contract_regression_table
)


executor_definition_summary = pd.Series(
    {
        "seeded_contracts_checked": len(
            execution_contract_regression_table
        ),
        "seeded_contract_hashes_matched": int(
            execution_contract_regression_table[
                "contract_hash_matches"
            ].sum()
        ),
        "strict_event_parser_reused": True,
        "corrected_response_validator_reused": True,
        "schema_validation_errors_preserved": True,
        "grounding_details_preserved": True,
        "stdout_budget_detection_enabled": True,
        "automatic_retry_enabled": False,
        "executor_defined": True,
        "ground_truth_loaded": False,
        "network_request_made": False,
    },
    name="value",
)

executor_definition_summary

,record_index,condition,contract_hash_matches
0,0,deterministic_text,True
1,0,structured,True
2,50,deterministic_text,True
3,50,structured,True
4,100,deterministic_text,True
5,100,structured,True
6,150,deterministic_text,True
7,150,structured,True
8,199,deterministic_text,True
9,199,structured,True


seeded_contracts_checked                  10
seeded_contract_hashes_matched            10
strict_event_parser_reused              True
corrected_response_validator_reused     True
schema_validation_errors_preserved      True
grounding_details_preserved             True
stdout_budget_detection_enabled         True
automatic_retry_enabled                False
executor_defined                        True
ground_truth_loaded                    False
network_request_made                   False
Name: value, dtype: object

In [25]:
BATCH_PLAN_BY_KEY = {
    (
        plan_entry["record_index"],
        plan_entry["condition"],
    ): plan_entry
    for plan_entry in batch_request_plan
}

assert len(BATCH_PLAN_BY_KEY) == 400


ESTABLISHED_CHECKPOINT_FIELDS = {
    "auto_approval_used",
    "backend",
    "budget_exhausted",
    "cache_read_tokens",
    "cache_write_tokens",
    "canonical_payload_sha256",
    "checkpoint_created_at_utc",
    "checkpoint_origin",
    "citation_count_valid",
    "condition",
    "distinct_feature_names_valid",
    "elapsed_seconds",
    "empty_temporary_directory_used",
    "error_message",
    "error_type",
    "event_count",
    "event_types",
    "execution_contract_sha256",
    "experiment_phase",
    "feature_set_id",
    "files_attached",
    "finish_reasons",
    "ground_truth_loaded",
    "grounding_valid",
    "input_tokens",
    "malformed_line_count",
    "maximum_model_output_tokens_supplied",
    "observed_values_match",
    "opencode_version",
    "output_tokens",
    "overall_response_valid",
    "predicted_label_valid",
    "prompt_sha256",
    "prompt_template_version",
    "pure_mode_used",
    "reasoning_tokens",
    "record_index",
    "reported_cost",
    "request_completed",
    "request_started_at_utc",
    "requested_model",
    "research_data_transmitted",
    "return_code",
    "sample_id",
    "sample_id_sent_to_model",
    "schema_structure_valid",
    "stderr_excerpt",
    "stderr_present",
    "step_finish_count",
    "subprocess_timeout_seconds",
    "supported_feature_names_valid",
    "total_tokens",
    "validation_error",
    "visible_json_parsed",
    "visible_response",
    "visible_response_character_count",
}


def read_batch_checkpoint(
    checkpoint_path: Path,
) -> dict:
    """Read one formal-batch checkpoint and validate its top-level type."""
    checkpoint = json.loads(
        checkpoint_path.read_text(
            encoding="utf-8"
        )
    )

    if not isinstance(checkpoint, dict):
        raise ValueError(
            "Checkpoint is not a JSON object: "
            f"{checkpoint_path}"
        )

    missing_fields = (
        ESTABLISHED_CHECKPOINT_FIELDS
        - set(checkpoint)
    )

    if missing_fields:
        raise ValueError(
            "Checkpoint is missing established fields: "
            f"{sorted(missing_fields)}"
        )

    return checkpoint


def verify_batch_checkpoint_matches_plan(
    checkpoint: dict,
    plan_entry: dict,
) -> dict:
    """
    Verify an existing checkpoint before it may be skipped.

    Failed or invalid experimental results are still valid evidence and
    must be skipped rather than automatically retried, provided their
    request identity and stored validation state remain internally
    consistent.
    """
    record_index = plan_entry[
        "record_index"
    ]

    condition = plan_entry[
        "condition"
    ]

    expected_execution_contract_sha256 = (
        stable_json_sha256(
            build_opencode_execution_contract(
                prompt_sha256=(
                    plan_entry[
                        "prompt_sha256"
                    ]
                )
            )
        )
    )

    expected_fields = {
        "record_index": record_index,
        "sample_id": (
            plan_entry["sample_id"]
        ),
        "condition": condition,
        "canonical_payload_sha256": (
            plan_entry[
                "canonical_payload_sha256"
            ]
        ),
        "prompt_sha256": (
            plan_entry["prompt_sha256"]
        ),
        "execution_contract_sha256": (
            expected_execution_contract_sha256
        ),
        "feature_set_id": "primary_46",
        "backend": "opencode",
        "opencode_version": (
            OPENCODE_VERSION
        ),
        "requested_model": (
            OPENCODE_MODEL_ROUTE
        ),
        "prompt_template_version": (
            OPENCODE_PROMPT_TEMPLATE_VERSION
        ),
        "subprocess_timeout_seconds": (
            OPENCODE_REQUEST_TIMEOUT_SECONDS
        ),
        "pure_mode_used": True,
        "empty_temporary_directory_used": True,
        "files_attached": False,
        "auto_approval_used": False,
        "maximum_model_output_tokens_supplied": False,
        "sample_id_sent_to_model": False,
        "ground_truth_loaded": False,
    }

    mismatches = {
        field_name: {
            "expected": expected_value,
            "observed": checkpoint.get(
                field_name
            ),
        }
        for field_name, expected_value
        in expected_fields.items()
        if checkpoint.get(field_name)
        != expected_value
    }

    if mismatches:
        raise ValueError(
            "Existing checkpoint does not match "
            f"the locked request plan: {mismatches}"
        )

    visible_response = checkpoint[
        "visible_response"
    ]

    if not isinstance(
        visible_response,
        str,
    ):
        raise ValueError(
            "visible_response must be a string."
        )

    if (
        checkpoint[
            "visible_response_character_count"
        ]
        != len(visible_response)
    ):
        raise ValueError(
            "visible_response_character_count "
            "does not match the saved response."
        )

    validation_reproduced = True

    if visible_response:
        grounding_reference = (
            extract_grounding_reference(
                structured_records[
                    record_index
                ]["model_input"]
            )
        )

        revalidation = (
            validate_model_response(
                visible_response=(
                    visible_response
                ),
                grounding_reference=(
                    grounding_reference
                ),
            )
        )

        for validation_field in (
            REGRESSION_VALIDATION_FIELDS
        ):
            if (
                checkpoint[
                    validation_field
                ]
                != revalidation[
                    validation_field
                ]
            ):
                raise ValueError(
                    "Stored validation result "
                    "cannot be reproduced for "
                    f"{record_index}, {condition}, "
                    f"field={validation_field!r}."
                )

        if (
            "schema_validation_errors"
            in checkpoint
            and checkpoint[
                "schema_validation_errors"
            ]
            != revalidation[
                "schema_validation_errors"
            ]
        ):
            raise ValueError(
                "Stored schema-validation details "
                "cannot be reproduced."
            )

        if (
            "grounding_details"
            in checkpoint
            and checkpoint[
                "grounding_details"
            ]
            != revalidation[
                "grounding_details"
            ]
        ):
            raise ValueError(
                "Stored grounding details "
                "cannot be reproduced."
            )

    else:
        # An empty response cannot legitimately claim successful
        # parsing, grounding or overall validity.
        invalid_true_fields = [
            validation_field
            for validation_field
            in REGRESSION_VALIDATION_FIELDS
            if checkpoint[
                validation_field
            ]
        ]

        if invalid_true_fields:
            raise ValueError(
                "Empty response has impossible "
                "positive validation fields: "
                f"{invalid_true_fields}"
            )

    # Checkpoints newly generated in the formal phase must preserve
    # the corrected validator's detailed evidence.
    if (
        checkpoint["experiment_phase"]
        == "formal_batch_inference"
    ):
        for detailed_field in [
            "schema_validation_errors",
            "grounding_details",
        ]:
            if detailed_field not in checkpoint:
                raise ValueError(
                    "Formal checkpoint is missing "
                    f"{detailed_field!r}."
                )

    return {
        "record_index": record_index,
        "sample_id": (
            plan_entry["sample_id"]
        ),
        "condition": condition,
        "request_completed": checkpoint[
            "request_completed"
        ],
        "overall_response_valid": (
            checkpoint[
                "overall_response_valid"
            ]
        ),
        "budget_exhausted": (
            checkpoint[
                "budget_exhausted"
            ]
        ),
        "validation_reproduced": (
            validation_reproduced
        ),
        "checkpoint_verified": True,
    }


def load_and_verify_batch_checkpoint(
    plan_entry: dict,
) -> dict:
    """
    Load one existing checkpoint and verify it before reuse or skipping.
    """
    checkpoint_path = (
        batch_checkpoint_path(
            record_index=(
                plan_entry["record_index"]
            ),
            condition=(
                plan_entry["condition"]
            ),
        )
    )

    if not checkpoint_path.is_file():
        raise FileNotFoundError(
            f"Checkpoint does not exist: "
            f"{checkpoint_path}"
        )

    checkpoint = read_batch_checkpoint(
        checkpoint_path
    )

    verify_batch_checkpoint_matches_plan(
        checkpoint=checkpoint,
        plan_entry=plan_entry,
    )

    return checkpoint


def build_formal_checkpoint_record(
    *,
    result: dict,
) -> dict:
    """
    Add formal-phase provenance to one executor result.
    """
    return {
        **result,
        "experiment_phase": (
            "formal_batch_inference"
        ),
        "checkpoint_origin": (
            "formal_batch_execution"
        ),
        "checkpoint_created_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
    }


def persist_formal_batch_result(
    *,
    result: dict,
    plan_entry: dict,
) -> dict:
    """
    Persist one completed attempt immediately and verify the saved file.

    This function never overwrites divergent evidence.
    """
    checkpoint_record = (
        build_formal_checkpoint_record(
            result=result
        )
    )

    verify_batch_checkpoint_matches_plan(
        checkpoint=checkpoint_record,
        plan_entry=plan_entry,
    )

    checkpoint_path = (
        batch_checkpoint_path(
            record_index=(
                plan_entry["record_index"]
            ),
            condition=(
                plan_entry["condition"]
            ),
        )
    )

    write_result = (
        write_checkpoint_atomically(
            destination_path=(
                checkpoint_path
            ),
            checkpoint=(
                checkpoint_record
            ),
        )
    )

    persisted_checkpoint = (
        load_and_verify_batch_checkpoint(
            plan_entry
        )
    )

    return {
        "checkpoint_action": (
            write_result[
                "checkpoint_action"
            ]
        ),
        "checkpoint_sha256": (
            write_result[
                "checkpoint_sha256"
            ]
        ),
        "checkpoint_path": (
            write_result[
                "checkpoint_path"
            ]
        ),
        "checkpoint": (
            persisted_checkpoint
        ),
    }


def execute_and_checkpoint_plan_entry(
    plan_entry: dict,
) -> dict:
    """
    Execute at most one request and immediately checkpoint its result.

    If a matching checkpoint already exists, it is validated and skipped,
    regardless of whether its experimental response succeeded or failed.
    This prevents automatic retries.
    """
    checkpoint_path = (
        batch_checkpoint_path(
            record_index=(
                plan_entry["record_index"]
            ),
            condition=(
                plan_entry["condition"]
            ),
        )
    )

    if checkpoint_path.exists():
        existing_checkpoint = (
            load_and_verify_batch_checkpoint(
                plan_entry
            )
        )

        return {
            "action": (
                "verified_existing_and_skipped"
            ),
            "checkpoint": (
                existing_checkpoint
            ),
        }

    record_index = plan_entry[
        "record_index"
    ]

    structured_record = (
        structured_records[
            record_index
        ]
    )

    text_record = (
        text_records[
            record_index
        ]
    )

    assert (
        structured_record["sample_id"]
        == text_record["sample_id"]
        == plan_entry["sample_id"]
    )

    assert (
        structured_record[
            "canonical_payload_sha256"
        ]
        == text_record[
            "canonical_payload_sha256"
        ]
        == plan_entry[
            "canonical_payload_sha256"
        ]
    )

    grounding_reference = (
        extract_grounding_reference(
            structured_record[
                "model_input"
            ]
        )
    )

    result = (
        run_formal_opencode_request(
            plan_entry=plan_entry,
            grounding_reference=(
                grounding_reference
            ),
        )
    )

    persisted_result = (
        persist_formal_batch_result(
            result=result,
            plan_entry=plan_entry,
        )
    )

    return {
        "action": (
            persisted_result[
                "checkpoint_action"
            ]
        ),
        "checkpoint": (
            persisted_result[
                "checkpoint"
            ]
        ),
    }


# Validate every checkpoint currently present before constructing the
# resumable missing-request list. This cell does not execute missing work.
existing_checkpoint_audit_rows = []
missing_plan_entries = []

for plan_entry in batch_request_plan:
    checkpoint_path = (
        batch_checkpoint_path(
            record_index=(
                plan_entry["record_index"]
            ),
            condition=(
                plan_entry["condition"]
            ),
        )
    )

    if checkpoint_path.exists():
        checkpoint = (
            load_and_verify_batch_checkpoint(
                plan_entry
            )
        )

        existing_checkpoint_audit_rows.append(
            {
                "request_plan_position": (
                    plan_entry[
                        "request_plan_position"
                    ]
                ),
                "record_index": (
                    plan_entry[
                        "record_index"
                    ]
                ),
                "sample_id": (
                    plan_entry["sample_id"]
                ),
                "condition": (
                    plan_entry["condition"]
                ),
                "request_completed": (
                    checkpoint[
                        "request_completed"
                    ]
                ),
                "overall_response_valid": (
                    checkpoint[
                        "overall_response_valid"
                    ]
                ),
                "budget_exhausted": (
                    checkpoint[
                        "budget_exhausted"
                    ]
                ),
                "checkpoint_verified": True,
            }
        )

    else:
        missing_plan_entries.append(
            plan_entry
        )


existing_checkpoint_audit_table = (
    pd.DataFrame(
        existing_checkpoint_audit_rows
    )
    .sort_values(
        "request_plan_position"
    )
    .reset_index(drop=True)
)


assert len(
    existing_checkpoint_audit_table
) == 10

assert len(
    missing_plan_entries
) == 390

assert existing_checkpoint_audit_table[
    "checkpoint_verified"
].all()


display(
    existing_checkpoint_audit_table
)


checkpoint_resume_summary = pd.Series(
    {
        "planned_request_count": len(
            batch_request_plan
        ),
        "existing_checkpoint_count": len(
            existing_checkpoint_audit_table
        ),
        "existing_checkpoints_verified": int(
            existing_checkpoint_audit_table[
                "checkpoint_verified"
            ].sum()
        ),
        "missing_checkpoint_count": len(
            missing_plan_entries
        ),
        "automatic_retry_of_existing_results": False,
        "single_request_executor_connected": True,
        "immediate_atomic_persistence_connected": True,
        "new_request_executed": False,
        "formal_checkpoint_directory_modified": False,
        "ground_truth_loaded": False,
        "network_request_made": False,
    },
    name="value",
)

checkpoint_resume_summary

,request_plan_position,record_index,sample_id,condition,request_completed,overall_response_valid,budget_exhausted,checkpoint_verified
0,0,0,pilot_001,structured,True,True,False,True
1,1,0,pilot_001,deterministic_text,True,True,False,True
2,100,50,pilot_051,structured,True,True,False,True
3,101,50,pilot_051,deterministic_text,True,True,False,True
4,200,100,pilot_101,structured,True,True,False,True
5,201,100,pilot_101,deterministic_text,True,True,False,True
6,300,150,pilot_151,structured,True,True,False,True
7,301,150,pilot_151,deterministic_text,True,True,False,True
8,398,199,pilot_200,deterministic_text,True,True,False,True
9,399,199,pilot_200,structured,True,True,False,True


planned_request_count                       400
existing_checkpoint_count                    10
existing_checkpoints_verified                10
missing_checkpoint_count                    390
automatic_retry_of_existing_results       False
single_request_executor_connected          True
immediate_atomic_persistence_connected     True
new_request_executed                      False
formal_checkpoint_directory_modified      False
ground_truth_loaded                       False
network_request_made                      False
Name: value, dtype: object

In [26]:
from concurrent.futures import (
    FIRST_COMPLETED,
    ThreadPoolExecutor,
    wait,
)


MAXIMUM_CONCURRENT_REQUESTS = 3


def scan_formal_batch_checkpoint_state() -> dict:
    """
    Rebuild the resumable state from the locked plan and checkpoints.

    Every existing checkpoint is validated before it is classified as
    completed. Missing entries retain their original request-plan order.
    """
    existing_records = []
    missing_entries = []

    ordered_plan = sorted(
        batch_request_plan,
        key=lambda entry: entry[
            "request_plan_position"
        ],
    )

    for plan_entry in ordered_plan:
        checkpoint_path = (
            batch_checkpoint_path(
                record_index=(
                    plan_entry[
                        "record_index"
                    ]
                ),
                condition=(
                    plan_entry[
                        "condition"
                    ]
                ),
            )
        )

        if not checkpoint_path.exists():
            missing_entries.append(
                plan_entry
            )
            continue

        checkpoint = (
            load_and_verify_batch_checkpoint(
                plan_entry
            )
        )

        existing_records.append(
            {
                "request_plan_position": (
                    plan_entry[
                        "request_plan_position"
                    ]
                ),
                "record_index": (
                    plan_entry[
                        "record_index"
                    ]
                ),
                "sample_id": (
                    plan_entry["sample_id"]
                ),
                "condition": (
                    plan_entry["condition"]
                ),
                "request_completed": (
                    checkpoint[
                        "request_completed"
                    ]
                ),
                "overall_response_valid": (
                    checkpoint[
                        "overall_response_valid"
                    ]
                ),
                "budget_exhausted": (
                    checkpoint[
                        "budget_exhausted"
                    ]
                ),
                "error_type": (
                    checkpoint[
                        "error_type"
                    ]
                ),
                "checkpoint_verified": True,
            }
        )

    if existing_records:
        existing_table = (
            pd.DataFrame(
                existing_records
            )
            .sort_values(
                "request_plan_position"
            )
            .reset_index(drop=True)
        )

    else:
        existing_table = pd.DataFrame(
            columns=[
                "request_plan_position",
                "record_index",
                "sample_id",
                "condition",
                "request_completed",
                "overall_response_valid",
                "budget_exhausted",
                "error_type",
                "checkpoint_verified",
            ]
        )

    assert (
        len(existing_table)
        + len(missing_entries)
        == 400
    )

    existing_keys = {
        (
            record[
                "record_index"
            ],
            record[
                "condition"
            ],
        )
        for record in existing_records
    }

    missing_keys = {
        (
            entry[
                "record_index"
            ],
            entry[
                "condition"
            ],
        )
        for entry in missing_entries
    }

    assert existing_keys.isdisjoint(
        missing_keys
    )

    assert (
        len(existing_keys)
        == len(existing_records)
    )

    assert (
        len(missing_keys)
        == len(missing_entries)
    )

    assert (
        len(
            existing_keys
            | missing_keys
        )
        == 400
    )

    missing_positions = [
        entry[
            "request_plan_position"
        ]
        for entry in missing_entries
    ]

    assert (
        missing_positions
        == sorted(missing_positions)
    )

    return {
        "existing_table": (
            existing_table
        ),
        "missing_plan_entries": (
            missing_entries
        ),
    }


def run_resumable_formal_batch(
    *,
    maximum_concurrent_requests: int = (
        MAXIMUM_CONCURRENT_REQUESTS
    ),
    resume_after_previous_budget_exhaustion: bool = False,
) -> dict:
    """
    Execute missing requests with bounded concurrency.

    Guarantees:
    - existing checkpoints are validated before being skipped;
    - only missing request keys are submitted;
    - submission follows request_plan_position order;
    - at most three requests run concurrently;
    - each result is checkpointed immediately;
    - failed or invalid results are not retried automatically;
    - quota/budget evidence stops new submissions;
    - already-running requests are allowed to finish.
    """
    if not isinstance(
        maximum_concurrent_requests,
        int,
    ):
        raise TypeError(
            "maximum_concurrent_requests "
            "must be an integer."
        )

    if not (
        1
        <= maximum_concurrent_requests
        <= 3
    ):
        raise ValueError(
            "maximum_concurrent_requests "
            "must be between 1 and 3."
        )

    initial_state = (
        scan_formal_batch_checkpoint_state()
    )

    initial_existing_table = (
        initial_state[
            "existing_table"
        ]
    )

    pending_entries = (
        initial_state[
            "missing_plan_entries"
        ]
    )

    previous_budget_exhaustion_count = 0

    if not initial_existing_table.empty:
        previous_budget_exhaustion_count = int(
            initial_existing_table[
                "budget_exhausted"
            ]
            .fillna(False)
            .sum()
        )

    # A historical quota checkpoint requires an explicit resume decision.
    # It is never automatically retried.
    if (
        previous_budget_exhaustion_count
        > 0
        and not (
            resume_after_previous_budget_exhaustion
        )
    ):
        return {
            "progress_table": (
                pd.DataFrame()
            ),
            "fatal_errors": [],
            "summary": pd.Series(
                {
                    "initial_existing_checkpoint_count": len(
                        initial_existing_table
                    ),
                    "initial_missing_checkpoint_count": len(
                        pending_entries
                    ),
                    "new_requests_submitted": 0,
                    "new_results_checkpointed": 0,
                    "maximum_observed_in_flight": 0,
                    "stopped_scheduling_new_work": True,
                    "stop_reason": (
                        "previous_budget_exhaustion_"
                        "requires_explicit_resume"
                    ),
                    "remaining_missing_checkpoint_count": len(
                        pending_entries
                    ),
                    "ground_truth_loaded": False,
                },
                name="value",
            ),
        }

    progress_rows = []
    fatal_errors = []

    in_flight = {}

    next_pending_index = 0
    submitted_count = 0
    checkpointed_count = 0
    maximum_observed_in_flight = 0

    stop_scheduling = False
    stop_reason = None

    def submit_next_request(
        executor,
    ) -> bool:
        nonlocal next_pending_index
        nonlocal submitted_count
        nonlocal maximum_observed_in_flight

        if (
            stop_scheduling
            or next_pending_index
            >= len(pending_entries)
        ):
            return False

        plan_entry = pending_entries[
            next_pending_index
        ]

        next_pending_index += 1

        future = executor.submit(
            execute_and_checkpoint_plan_entry,
            plan_entry,
        )

        in_flight[
            future
        ] = plan_entry

        submitted_count += 1

        maximum_observed_in_flight = max(
            maximum_observed_in_flight,
            len(in_flight),
        )

        return True

    print(
        "Formal batch starting: "
        f"verified_existing="
        f"{len(initial_existing_table)}, "
        f"missing="
        f"{len(pending_entries)}, "
        f"maximum_concurrency="
        f"{maximum_concurrent_requests}",
        flush=True,
    )

    with ThreadPoolExecutor(
        max_workers=(
            maximum_concurrent_requests
        ),
        thread_name_prefix=(
            "compsci742_batch"
        ),
    ) as executor:
        # Fill the initial bounded worker pool.
        while (
            len(in_flight)
            < maximum_concurrent_requests
        ):
            if not submit_next_request(
                executor
            ):
                break

        while in_flight:
            completed_futures, _ = wait(
                set(in_flight),
                return_when=(
                    FIRST_COMPLETED
                ),
            )

            # When several finish together, process their evidence in
            # original request-plan order.
            ordered_completed_futures = sorted(
                completed_futures,
                key=lambda future: (
                    in_flight[
                        future
                    ][
                        "request_plan_position"
                    ]
                ),
            )

            for future in (
                ordered_completed_futures
            ):
                plan_entry = (
                    in_flight.pop(
                        future
                    )
                )

                try:
                    outcome = (
                        future.result()
                    )

                    checkpoint = (
                        outcome[
                            "checkpoint"
                        ]
                    )

                    checkpointed_count += 1

                    progress_rows.append(
                        {
                            "request_plan_position": (
                                plan_entry[
                                    "request_plan_position"
                                ]
                            ),
                            "record_index": (
                                plan_entry[
                                    "record_index"
                                ]
                            ),
                            "sample_id": (
                                plan_entry[
                                    "sample_id"
                                ]
                            ),
                            "condition": (
                                plan_entry[
                                    "condition"
                                ]
                            ),
                            "action": (
                                outcome[
                                    "action"
                                ]
                            ),
                            "request_completed": (
                                checkpoint[
                                    "request_completed"
                                ]
                            ),
                            "return_code": (
                                checkpoint[
                                    "return_code"
                                ]
                            ),
                            "overall_response_valid": (
                                checkpoint[
                                    "overall_response_valid"
                                ]
                            ),
                            "budget_exhausted": (
                                checkpoint[
                                    "budget_exhausted"
                                ]
                            ),
                            "elapsed_seconds": (
                                checkpoint[
                                    "elapsed_seconds"
                                ]
                            ),
                            "error_type": (
                                checkpoint[
                                    "error_type"
                                ]
                            ),
                        }
                    )

                    print(
                        f"checkpointed="
                        f"{checkpointed_count}/"
                        f"{len(pending_entries)} "
                        f"position="
                        f"{plan_entry['request_plan_position']} "
                        f"condition="
                        f"{plan_entry['condition']} "
                        f"valid="
                        f"{checkpoint['overall_response_valid']} "
                        f"budget_exhausted="
                        f"{checkpoint['budget_exhausted']}",
                        flush=True,
                    )

                    if checkpoint[
                        "budget_exhausted"
                    ]:
                        stop_scheduling = True

                        if stop_reason is None:
                            stop_reason = (
                                "budget_or_quota_"
                                "exhausted"
                            )

                except Exception as exc:
                    # This path represents an unexpected scheduler,
                    # verification or persistence problem—not an ordinary
                    # invalid model response.
                    stop_scheduling = True

                    if stop_reason is None:
                        stop_reason = (
                            "unexpected_scheduler_"
                            "or_persistence_error"
                        )

                    fatal_errors.append(
                        {
                            "request_plan_position": (
                                plan_entry[
                                    "request_plan_position"
                                ]
                            ),
                            "record_index": (
                                plan_entry[
                                    "record_index"
                                ]
                            ),
                            "sample_id": (
                                plan_entry[
                                    "sample_id"
                                ]
                            ),
                            "condition": (
                                plan_entry[
                                    "condition"
                                ]
                            ),
                            "error_type": (
                                type(exc).__name__
                            ),
                            "error_message": (
                                str(exc)[:500]
                            ),
                        }
                    )

                    print(
                        "FATAL: stopped new "
                        "submissions after "
                        f"position="
                        f"{plan_entry['request_plan_position']} "
                        f"error="
                        f"{type(exc).__name__}: "
                        f"{str(exc)[:300]}",
                        flush=True,
                    )

            # Replenish workers only when no stop condition was observed.
            if not stop_scheduling:
                while (
                    len(in_flight)
                    < maximum_concurrent_requests
                ):
                    if not submit_next_request(
                        executor
                    ):
                        break

    # All in-flight work has now finished and persisted or surfaced a
    # fatal persistence error. Rebuild state from disk.
    final_state = (
        scan_formal_batch_checkpoint_state()
    )

    final_existing_table = (
        final_state[
            "existing_table"
        ]
    )

    final_missing_entries = (
        final_state[
            "missing_plan_entries"
        ]
    )

    if progress_rows:
        progress_table = (
            pd.DataFrame(
                progress_rows
            )
            .sort_values(
                "request_plan_position"
            )
            .reset_index(drop=True)
        )

    else:
        progress_table = (
            pd.DataFrame()
        )

    if stop_reason is None:
        stop_reason = (
            "all_available_missing_"
            "requests_submitted"
        )

    summary = pd.Series(
        {
            "initial_existing_checkpoint_count": len(
                initial_existing_table
            ),
            "initial_missing_checkpoint_count": len(
                pending_entries
            ),
            "new_requests_submitted": (
                submitted_count
            ),
            "new_results_checkpointed": (
                checkpointed_count
            ),
            "maximum_observed_in_flight": (
                maximum_observed_in_flight
            ),
            "fatal_error_count": len(
                fatal_errors
            ),
            "stopped_scheduling_new_work": (
                stop_scheduling
            ),
            "stop_reason": (
                stop_reason
            ),
            "final_existing_checkpoint_count": len(
                final_existing_table
            ),
            "remaining_missing_checkpoint_count": len(
                final_missing_entries
            ),
            "automatic_retry_used": False,
            "ground_truth_loaded": False,
        },
        name="value",
    )

    return {
        "progress_table": (
            progress_table
        ),
        "fatal_errors": (
            fatal_errors
        ),
        "summary": summary,
    }


# Dry-run readiness check only.
# run_resumable_formal_batch() is NOT called in this cell.
scheduler_dry_run_state = (
    scan_formal_batch_checkpoint_state()
)

scheduler_dry_run_existing = (
    scheduler_dry_run_state[
        "existing_table"
    ]
)

scheduler_dry_run_missing = (
    scheduler_dry_run_state[
        "missing_plan_entries"
    ]
)

scheduler_dry_run_preview = (
    pd.DataFrame(
        [
            {
                "request_plan_position": (
                    entry[
                        "request_plan_position"
                    ]
                ),
                "record_index": (
                    entry[
                        "record_index"
                    ]
                ),
                "sample_id": (
                    entry["sample_id"]
                ),
                "condition": (
                    entry["condition"]
                ),
                "order_group": (
                    entry["order_group"]
                ),
                "within_pair_position": (
                    entry[
                        "within_pair_position"
                    ]
                ),
            }
            for entry in (
                scheduler_dry_run_missing[
                    :12
                ]
            )
        ]
    )
)


assert (
    MAXIMUM_CONCURRENT_REQUESTS
    == 3
)

assert (
    len(
        scheduler_dry_run_existing
    )
    == 10
)

assert (
    len(
        scheduler_dry_run_missing
    )
    == 390
)

assert sum(
    entry["condition"]
    == "structured"
    for entry in (
        scheduler_dry_run_missing
    )
) == 195

assert sum(
    entry["condition"]
    == "deterministic_text"
    for entry in (
        scheduler_dry_run_missing
    )
) == 195

assert [
    entry[
        "request_plan_position"
    ]
    for entry in (
        scheduler_dry_run_missing
    )
] == sorted(
    entry[
        "request_plan_position"
    ]
    for entry in (
        scheduler_dry_run_missing
    )
)


display(
    scheduler_dry_run_preview
)


scheduler_definition_summary = pd.Series(
    {
        "maximum_concurrent_requests": (
            MAXIMUM_CONCURRENT_REQUESTS
        ),
        "verified_existing_checkpoint_count": len(
            scheduler_dry_run_existing
        ),
        "dry_run_missing_request_count": len(
            scheduler_dry_run_missing
        ),
        "dry_run_structured_request_count": sum(
            entry["condition"]
            == "structured"
            for entry in (
                scheduler_dry_run_missing
            )
        ),
        "dry_run_deterministic_text_request_count": sum(
            entry["condition"]
            == "deterministic_text"
            for entry in (
                scheduler_dry_run_missing
            )
        ),
        "precomputed_order_retained": True,
        "existing_checkpoints_validated_before_skip": True,
        "failed_or_invalid_results_automatically_retried": False,
        "budget_exhaustion_stops_new_submissions": True,
        "already_running_requests_allowed_to_finish": True,
        "scheduler_defined": True,
        "new_request_executed": False,
        "formal_checkpoint_directory_modified": False,
        "ground_truth_loaded": False,
        "network_request_made": False,
    },
    name="value",
)

scheduler_definition_summary

,request_plan_position,record_index,sample_id,condition,order_group,within_pair_position
0,2,1,pilot_002,deterministic_text,deterministic_text_first,1
1,3,1,pilot_002,structured,deterministic_text_first,2
2,4,2,pilot_003,structured,structured_first,1
3,5,2,pilot_003,deterministic_text,structured_first,2
4,6,3,pilot_004,deterministic_text,deterministic_text_first,1
5,7,3,pilot_004,structured,deterministic_text_first,2
6,8,4,pilot_005,structured,structured_first,1
7,9,4,pilot_005,deterministic_text,structured_first,2
8,10,5,pilot_006,deterministic_text,deterministic_text_first,1
9,11,5,pilot_006,structured,deterministic_text_first,2


maximum_concurrent_requests                            3
verified_existing_checkpoint_count                    10
dry_run_missing_request_count                        390
dry_run_structured_request_count                     195
dry_run_deterministic_text_request_count             195
precomputed_order_retained                          True
existing_checkpoints_validated_before_skip          True
failed_or_invalid_results_automatically_retried    False
budget_exhaustion_stops_new_submissions             True
already_running_requests_allowed_to_finish          True
scheduler_defined                                   True
new_request_executed                               False
formal_checkpoint_directory_modified               False
ground_truth_loaded                                False
network_request_made                               False
Name: value, dtype: object

In [27]:
SMOKE_REQUEST_PLAN_POSITION = 2


smoke_plan_matches = [
    plan_entry
    for plan_entry in batch_request_plan
    if plan_entry["request_plan_position"]
    == SMOKE_REQUEST_PLAN_POSITION
]

assert len(smoke_plan_matches) == 1

smoke_plan_entry = smoke_plan_matches[0]

assert smoke_plan_entry["reusable_checkpoint_available"] is False
assert smoke_plan_entry["sample_id"] not in smoke_plan_entry["prompt"]

smoke_checkpoint_path = batch_checkpoint_path(
    record_index=smoke_plan_entry["record_index"],
    condition=smoke_plan_entry["condition"],
)

smoke_checkpoint_existed_before = smoke_checkpoint_path.exists()
smoke_state_before = scan_formal_batch_checkpoint_state()
smoke_existing_count_before = len(smoke_state_before["existing_table"])
smoke_missing_count_before = len(
    smoke_state_before["missing_plan_entries"]
)

smoke_outcome = execute_and_checkpoint_plan_entry(
    smoke_plan_entry
)
smoke_checkpoint = smoke_outcome["checkpoint"]

assert smoke_checkpoint_path.is_file()

# Read and independently verify the persisted checkpoint once more.
smoke_persisted_checkpoint = load_and_verify_batch_checkpoint(
    smoke_plan_entry
)

assert smoke_persisted_checkpoint == smoke_checkpoint
assert smoke_persisted_checkpoint["experiment_phase"] == (
    "formal_batch_inference"
)
assert smoke_persisted_checkpoint["checkpoint_origin"] == (
    "formal_batch_execution"
)
assert "schema_validation_errors" in smoke_persisted_checkpoint
assert "grounding_details" in smoke_persisted_checkpoint
assert smoke_persisted_checkpoint["ground_truth_loaded"] is False
assert (
    smoke_persisted_checkpoint["sample_id_sent_to_model"]
    is False
)
assert smoke_persisted_checkpoint["files_attached"] is False
assert (
    smoke_persisted_checkpoint["auto_approval_used"]
    is False
)
assert smoke_persisted_checkpoint[
    "maximum_model_output_tokens_supplied"
] is False

smoke_expected_execution_contract_sha256 = stable_json_sha256(
    build_opencode_execution_contract(
        prompt_sha256=smoke_plan_entry["prompt_sha256"]
    )
)

assert smoke_persisted_checkpoint[
    "execution_contract_sha256"
] == smoke_expected_execution_contract_sha256

smoke_state_after = scan_formal_batch_checkpoint_state()
smoke_existing_count_after = len(
    smoke_state_after["existing_table"]
)
smoke_missing_count_after = len(
    smoke_state_after["missing_plan_entries"]
)

if smoke_checkpoint_existed_before:
    assert (
        smoke_outcome["action"]
        == "verified_existing_and_skipped"
    )
    assert (
        smoke_existing_count_after
        == smoke_existing_count_before
    )
    assert (
        smoke_missing_count_after
        == smoke_missing_count_before
    )
else:
    assert smoke_outcome["action"] == "created"
    assert (
        smoke_existing_count_after
        == smoke_existing_count_before + 1
    )
    assert (
        smoke_missing_count_after
        == smoke_missing_count_before - 1
    )

smoke_operational_gate_passed = all(
    [
        smoke_persisted_checkpoint["request_completed"] is True,
        smoke_persisted_checkpoint["return_code"] == 0,
        smoke_persisted_checkpoint["budget_exhausted"] is False,
        smoke_persisted_checkpoint["error_type"] is None,
        smoke_persisted_checkpoint["error_message"] is None,
        smoke_persisted_checkpoint["malformed_line_count"] == 0,
        smoke_persisted_checkpoint["step_finish_count"] >= 1,
    ]
)

smoke_checkpoint_summary = pd.Series(
    {
        "fixed_request_plan_position": (
            SMOKE_REQUEST_PLAN_POSITION
        ),
        "record_index": smoke_plan_entry["record_index"],
        "sample_id": smoke_plan_entry["sample_id"],
        "condition": smoke_plan_entry["condition"],
        "checkpoint_existed_before": (
            smoke_checkpoint_existed_before
        ),
        "action": smoke_outcome["action"],
        "request_completed": smoke_persisted_checkpoint[
            "request_completed"
        ],
        "return_code": smoke_persisted_checkpoint[
            "return_code"
        ],
        "overall_response_valid": smoke_persisted_checkpoint[
            "overall_response_valid"
        ],
        "schema_structure_valid": smoke_persisted_checkpoint[
            "schema_structure_valid"
        ],
        "grounding_valid": smoke_persisted_checkpoint[
            "grounding_valid"
        ],
        "schema_validation_error_count": len(
            smoke_persisted_checkpoint[
                "schema_validation_errors"
            ]
        ),
        "grounding_details_preserved": (
            smoke_persisted_checkpoint[
                "grounding_details"
            ]
            is not None
        ),
        "budget_exhausted": smoke_persisted_checkpoint[
            "budget_exhausted"
        ],
        "error_type": smoke_persisted_checkpoint[
            "error_type"
        ],
        "elapsed_seconds": smoke_persisted_checkpoint[
            "elapsed_seconds"
        ],
        "total_tokens": smoke_persisted_checkpoint[
            "total_tokens"
        ],
        "checkpoint_sha256": sha256_file(
            smoke_checkpoint_path
        ),
        "existing_checkpoint_count_before": (
            smoke_existing_count_before
        ),
        "existing_checkpoint_count_after": (
            smoke_existing_count_after
        ),
        "remaining_missing_checkpoint_count": (
            smoke_missing_count_after
        ),
        "operational_gate_passed": (
            smoke_operational_gate_passed
        ),
        "network_request_made_by_this_cell": (
            not smoke_checkpoint_existed_before
            and smoke_persisted_checkpoint[
                "research_data_transmitted"
            ]
        ),
        "automatic_retry_used": False,
        "ground_truth_loaded": False,
    },
    name="value",
)

smoke_checkpoint_summary

fixed_request_plan_position                                                           2
record_index                                                                          1
sample_id                                                                     pilot_002
condition                                                            deterministic_text
checkpoint_existed_before                                                         False
action                                                                          created
request_completed                                                                  True
return_code                                                                           0
overall_response_valid                                                             True
schema_structure_valid                                                             True
grounding_valid                                                                    True
schema_validation_error_count   

In [28]:
formal_batch_state_before = scan_formal_batch_checkpoint_state()
formal_batch_existing_before = formal_batch_state_before[
    "existing_table"
]
formal_batch_missing_before = formal_batch_state_before[
    "missing_plan_entries"
]

assert (
    len(formal_batch_existing_before)
    + len(formal_batch_missing_before)
    == 400
)
assert MAXIMUM_CONCURRENT_REQUESTS == 3

formal_batch_run = run_resumable_formal_batch(
    maximum_concurrent_requests=MAXIMUM_CONCURRENT_REQUESTS,
)

formal_batch_progress_table = formal_batch_run[
    "progress_table"
]
formal_batch_fatal_errors = formal_batch_run["fatal_errors"]
formal_batch_scheduler_summary = formal_batch_run["summary"]

formal_batch_state_after = scan_formal_batch_checkpoint_state()
formal_batch_existing_after = formal_batch_state_after[
    "existing_table"
]
formal_batch_missing_after = formal_batch_state_after[
    "missing_plan_entries"
]

assert (
    len(formal_batch_existing_after)
    + len(formal_batch_missing_after)
    == 400
)
assert len(formal_batch_progress_table) == int(
    formal_batch_scheduler_summary["new_results_checkpointed"]
)

formal_batch_completed_count = int(
    formal_batch_existing_after["request_completed"]
    .eq(True)
    .sum()
)
formal_batch_valid_count = int(
    formal_batch_existing_after["overall_response_valid"]
    .eq(True)
    .sum()
)
formal_batch_budget_exhausted_count = int(
    formal_batch_existing_after["budget_exhausted"]
    .fillna(False)
    .eq(True)
    .sum()
)
formal_batch_error_count = int(
    formal_batch_existing_after["error_type"]
    .notna()
    .sum()
)

formal_batch_all_checkpointed = (
    len(formal_batch_existing_after) == 400
    and len(formal_batch_missing_after) == 0
)

formal_batch_post_run_summary = pd.Series(
    {
        "initial_verified_checkpoint_count": len(
            formal_batch_existing_before
        ),
        "initial_missing_checkpoint_count": len(
            formal_batch_missing_before
        ),
        "new_requests_submitted_this_run": int(
            formal_batch_scheduler_summary[
                "new_requests_submitted"
            ]
        ),
        "new_results_checkpointed_this_run": int(
            formal_batch_scheduler_summary[
                "new_results_checkpointed"
            ]
        ),
        "maximum_observed_in_flight": int(
            formal_batch_scheduler_summary[
                "maximum_observed_in_flight"
            ]
        ),
        "scheduler_stop_reason": (
            formal_batch_scheduler_summary["stop_reason"]
        ),
        "final_verified_checkpoint_count": len(
            formal_batch_existing_after
        ),
        "remaining_missing_checkpoint_count": len(
            formal_batch_missing_after
        ),
        "request_completed_checkpoint_count": (
            formal_batch_completed_count
        ),
        "overall_valid_checkpoint_count": (
            formal_batch_valid_count
        ),
        "invalid_checkpoint_count": (
            len(formal_batch_existing_after)
            - formal_batch_valid_count
        ),
        "budget_exhausted_checkpoint_count": (
            formal_batch_budget_exhausted_count
        ),
        "checkpoint_error_count": formal_batch_error_count,
        "fatal_scheduler_error_count": len(
            formal_batch_fatal_errors
        ),
        "all_400_requests_checkpointed": (
            formal_batch_all_checkpointed
        ),
        "ready_for_aggregation": (
            formal_batch_all_checkpointed
            and len(formal_batch_fatal_errors) == 0
        ),
        "automatic_retry_used": False,
        "ground_truth_loaded": False,
    },
    name="value",
)

display(formal_batch_scheduler_summary)

if not formal_batch_progress_table.empty:
    display(formal_batch_progress_table.tail(12))

if formal_batch_fatal_errors:
    display(pd.DataFrame(formal_batch_fatal_errors))

formal_batch_post_run_summary

Formal batch starting: verified_existing=11, missing=389, maximum_concurrency=3
checkpointed=1/389 position=5 condition=deterministic_text valid=True budget_exhausted=False
checkpointed=2/389 position=6 condition=deterministic_text valid=True budget_exhausted=False
checkpointed=3/389 position=3 condition=structured valid=True budget_exhausted=False
checkpointed=4/389 position=4 condition=structured valid=True budget_exhausted=False
checkpointed=5/389 position=7 condition=structured valid=True budget_exhausted=False
checkpointed=6/389 position=8 condition=structured valid=True budget_exhausted=False
checkpointed=7/389 position=10 condition=deterministic_text valid=True budget_exhausted=False
checkpointed=8/389 position=11 condition=structured valid=True budget_exhausted=False
checkpointed=9/389 position=12 condition=structured valid=True budget_exhausted=False
checkpointed=10/389 position=14 condition=deterministic_text valid=True budget_exhausted=False
checkpointed=11/389 position=13 c

initial_existing_checkpoint_count                                           11
initial_missing_checkpoint_count                                           389
new_requests_submitted                                                     389
new_results_checkpointed                                                   389
maximum_observed_in_flight                                                   3
fatal_error_count                                                            0
stopped_scheduling_new_work                                              False
stop_reason                           all_available_missing_requests_submitted
final_existing_checkpoint_count                                            400
remaining_missing_checkpoint_count                                           0
automatic_retry_used                                                     False
ground_truth_loaded                                                      False
Name: value, dtype: object

,request_plan_position,record_index,sample_id,condition,action,request_completed,return_code,overall_response_valid,budget_exhausted,elapsed_seconds,error_type
377,386,193,pilot_194,deterministic_text,created,False,NaN,False,False,300.042374,TimeoutExpired
378,387,193,pilot_194,structured,created,False,NaN,False,False,300.040083,TimeoutExpired
379,388,194,pilot_195,structured,created,False,NaN,False,False,300.032513,TimeoutExpired
380,389,194,pilot_195,deterministic_text,created,False,NaN,False,False,300.122511,TimeoutExpired
381,390,195,pilot_196,deterministic_text,created,False,NaN,False,False,300.047631,TimeoutExpired
382,391,195,pilot_196,structured,created,False,NaN,False,False,300.069637,TimeoutExpired
383,392,196,pilot_197,structured,created,False,NaN,False,False,300.129634,TimeoutExpired
384,393,196,pilot_197,deterministic_text,created,False,NaN,False,False,300.061733,TimeoutExpired
385,394,197,pilot_198,deterministic_text,created,False,NaN,False,False,300.072706,TimeoutExpired
386,395,197,pilot_198,structured,created,False,NaN,False,False,300.073035,TimeoutExpired


initial_verified_checkpoint_count                                           11
initial_missing_checkpoint_count                                           389
new_requests_submitted_this_run                                            389
new_results_checkpointed_this_run                                          389
maximum_observed_in_flight                                                   3
scheduler_stop_reason                 all_available_missing_requests_submitted
final_verified_checkpoint_count                                            400
remaining_missing_checkpoint_count                                           0
request_completed_checkpoint_count                                         360
overall_valid_checkpoint_count                                             359
invalid_checkpoint_count                                                    41
budget_exhausted_checkpoint_count                                            0
checkpoint_error_count                              

In [29]:
POST_BATCH_VALIDATION_FIELDS = [
    "visible_json_parsed",
    "schema_structure_valid",
    "predicted_label_valid",
    "citation_count_valid",
    "supported_feature_names_valid",
    "distinct_feature_names_valid",
    "observed_values_match",
    "grounding_valid",
    "overall_response_valid",
]


post_batch_audit_rows = []

for plan_entry in sorted(
    batch_request_plan,
    key=lambda entry: entry["request_plan_position"],
):
    checkpoint = load_and_verify_batch_checkpoint(plan_entry)

    audit_row = {
        "request_plan_position": plan_entry[
            "request_plan_position"
        ],
        "record_index": plan_entry["record_index"],
        "sample_id": plan_entry["sample_id"],
        "condition": plan_entry["condition"],
        "checkpoint_origin": checkpoint["checkpoint_origin"],
        "request_completed": checkpoint["request_completed"],
        "return_code": checkpoint["return_code"],
        "budget_exhausted": checkpoint["budget_exhausted"],
        "error_type": checkpoint["error_type"],
        "ground_truth_loaded": checkpoint["ground_truth_loaded"],
    }

    for field_name in POST_BATCH_VALIDATION_FIELDS:
        audit_row[field_name] = checkpoint[field_name]

    post_batch_audit_rows.append(audit_row)


post_batch_audit_table = pd.DataFrame(
    post_batch_audit_rows
)

assert len(post_batch_audit_table) == 400
assert post_batch_audit_table[
    ["record_index", "condition"]
].drop_duplicates().shape[0] == 400
assert post_batch_audit_table["sample_id"].nunique() == 200
assert (
    post_batch_audit_table["condition"]
    .value_counts()
    .to_dict()
    == {
        "structured": 200,
        "deterministic_text": 200,
    }
)
assert post_batch_audit_table[
    "ground_truth_loaded"
].eq(False).all()
assert post_batch_audit_table[
    "budget_exhausted"
].eq(False).all()


post_batch_condition_summary = (
    post_batch_audit_table
    .groupby("condition", sort=True)
    .agg(
        checkpoint_count=("sample_id", "size"),
        request_completed_count=("request_completed", "sum"),
        visible_json_parsed_count=(
            "visible_json_parsed",
            "sum",
        ),
        schema_valid_count=("schema_structure_valid", "sum"),
        grounding_valid_count=("grounding_valid", "sum"),
        overall_valid_count=("overall_response_valid", "sum"),
        budget_exhausted_count=("budget_exhausted", "sum"),
        execution_error_count=(
            "error_type",
            lambda values: int(values.notna().sum()),
        ),
    )
    .reset_index()
)

post_batch_condition_summary["invalid_count"] = (
    post_batch_condition_summary["checkpoint_count"]
    - post_batch_condition_summary["overall_valid_count"]
)


post_batch_error_type_summary = (
    post_batch_audit_table.loc[
        post_batch_audit_table["error_type"].notna(),
        ["condition", "error_type"],
    ]
    .groupby(
        ["condition", "error_type"],
        sort=True,
    )
    .size()
    .rename("checkpoint_count")
    .reset_index()
)


post_batch_validation_failure_summary = pd.DataFrame(
    [
        {
            "validation_field": field_name,
            "failed_checkpoint_count": int(
                post_batch_audit_table[field_name]
                .eq(False)
                .sum()
            ),
        }
        for field_name in POST_BATCH_VALIDATION_FIELDS
    ]
)


post_batch_completed_invalid_table = (
    post_batch_audit_table.loc[
        post_batch_audit_table["request_completed"].eq(True)
        & post_batch_audit_table[
            "overall_response_valid"
        ].eq(False),
        [
            "request_plan_position",
            "record_index",
            "sample_id",
            "condition",
            "return_code",
            "error_type",
            *POST_BATCH_VALIDATION_FIELDS,
        ],
    ]
    .sort_values("request_plan_position")
    .reset_index(drop=True)
)


post_batch_operational_error_summary = (
    post_batch_audit_table.loc[
        post_batch_audit_table["error_type"].notna(),
        [
            "condition",
            "request_completed",
            "return_code",
            "error_type",
        ],
    ]
    .groupby(
        [
            "condition",
            "request_completed",
            "return_code",
            "error_type",
        ],
        dropna=False,
        sort=True,
    )
    .size()
    .rename("checkpoint_count")
    .reset_index()
)


post_batch_audit_summary = pd.Series(
    {
        "checkpoint_count": len(post_batch_audit_table),
        "paired_sample_count": (
            post_batch_audit_table["sample_id"].nunique()
        ),
        "request_completed_count": int(
            post_batch_audit_table[
                "request_completed"
            ].sum()
        ),
        "overall_valid_count": int(
            post_batch_audit_table[
                "overall_response_valid"
            ].sum()
        ),
        "invalid_count": int(
            post_batch_audit_table[
                "overall_response_valid"
            ].eq(False).sum()
        ),
        "execution_error_count": int(
            post_batch_audit_table[
                "error_type"
            ].notna().sum()
        ),
        "completed_but_invalid_count": len(
            post_batch_completed_invalid_table
        ),
        "budget_exhausted_count": int(
            post_batch_audit_table[
                "budget_exhausted"
            ].sum()
        ),
        "all_checkpoint_identities_reverified": True,
        "automatic_retry_used": False,
        "network_request_made_by_this_cell": False,
        "ground_truth_loaded": False,
    },
    name="value",
)


display(post_batch_condition_summary)
display(post_batch_error_type_summary)
display(post_batch_validation_failure_summary)
display(post_batch_completed_invalid_table)
display(post_batch_operational_error_summary)

post_batch_audit_summary

,condition,checkpoint_count,request_completed_count,visible_json_parsed_count,schema_valid_count,grounding_valid_count,overall_valid_count,budget_exhausted_count,execution_error_count,invalid_count
0,deterministic_text,200,181,181,180,180,180,0,19,20
1,structured,200,179,179,179,179,179,0,21,21


,condition,error_type,checkpoint_count
0,deterministic_text,TimeoutExpired,19
1,structured,TimeoutExpired,21


,validation_field,failed_checkpoint_count
0,visible_json_parsed,40
1,schema_structure_valid,41
2,predicted_label_valid,41
3,citation_count_valid,41
4,supported_feature_names_valid,41
5,distinct_feature_names_valid,41
6,observed_values_match,41
7,grounding_valid,41
8,overall_response_valid,41


,request_plan_position,record_index,sample_id,condition,return_code,error_type,visible_json_parsed,schema_structure_valid,predicted_label_valid,citation_count_valid,supported_feature_names_valid,distinct_feature_names_valid,observed_values_match,grounding_valid,overall_response_valid
0,13,6,pilot_007,deterministic_text,0.0,NaN,True,False,False,False,False,False,False,False,False


,condition,request_completed,return_code,error_type,checkpoint_count
0,deterministic_text,False,NaN,TimeoutExpired,19
1,structured,False,NaN,TimeoutExpired,21


checkpoint_count                          400
paired_sample_count                       200
request_completed_count                   360
overall_valid_count                       359
invalid_count                              41
execution_error_count                      40
completed_but_invalid_count                 1
budget_exhausted_count                      0
all_checkpoint_identities_reverified     True
automatic_retry_used                    False
network_request_made_by_this_cell       False
ground_truth_loaded                     False
Name: value, dtype: object

In [30]:
from collections import Counter


BATCH_RESULTS_JSONL_PATH = (
    BATCH_RESULTS_DIR / "opencode_batch_results.jsonl"
)
BATCH_MANIFEST_PATH = (
    BATCH_RESULTS_DIR / "opencode_batch_manifest.json"
)


def project_relative_path(path: Path) -> str:
    """Return a stable project-relative POSIX path."""
    return path.resolve().relative_to(
        PROJECT_ROOT.resolve()
    ).as_posix()


def write_artifact_atomically_without_overwrite(
    destination_path: Path,
    content: bytes,
) -> str:
    """Create an artifact atomically; never overwrite divergent evidence."""
    destination_path.parent.mkdir(parents=True, exist_ok=True)

    if destination_path.exists():
        if destination_path.read_bytes() != content:
            raise RuntimeError(
                "Refusing to overwrite divergent artifact: "
                f"{destination_path}"
            )
        return "verified_existing"

    temporary_path = destination_path.with_name(
        destination_path.name + ".tmp"
    )

    if temporary_path.exists():
        raise RuntimeError(
            f"Unexpected temporary artifact: {temporary_path}"
        )

    temporary_path.write_bytes(content)

    if temporary_path.read_bytes() != content:
        raise RuntimeError(
            "Temporary artifact verification failed: "
            f"{temporary_path}"
        )

    temporary_path.replace(destination_path)

    if destination_path.read_bytes() != content:
        raise RuntimeError(
            f"Final artifact verification failed: {destination_path}"
        )

    return "created"


ordered_batch_plan = sorted(
    batch_request_plan,
    key=lambda entry: entry["request_plan_position"],
)

assert len(ordered_batch_plan) == 400
assert [
    entry["request_plan_position"]
    for entry in ordered_batch_plan
] == list(range(400))


aggregate_records = []
checkpoint_inventory = []

for plan_entry in ordered_batch_plan:
    checkpoint_path = batch_checkpoint_path(
        record_index=plan_entry["record_index"],
        condition=plan_entry["condition"],
    )

    if not checkpoint_path.is_file():
        raise FileNotFoundError(
            f"Missing formal checkpoint: {checkpoint_path}"
        )

    checkpoint = load_and_verify_batch_checkpoint(
        plan_entry
    )
    checkpoint_sha256 = sha256_file(checkpoint_path)
    aggregate_record = dict(checkpoint)

    aggregate_metadata = {
        "request_plan_position": plan_entry[
            "request_plan_position"
        ],
        "order_group": plan_entry["order_group"],
        "within_pair_position": plan_entry[
            "within_pair_position"
        ],
        "checkpoint_relative_path": project_relative_path(
            checkpoint_path
        ),
        "checkpoint_file_sha256": checkpoint_sha256,
    }

    for field_name, expected_value in (
        aggregate_metadata.items()
    ):
        if (
            field_name in aggregate_record
            and aggregate_record[field_name]
            != expected_value
        ):
            raise RuntimeError(
                f"Conflicting aggregate field {field_name}: "
                f"{checkpoint_path}"
            )

        aggregate_record[field_name] = expected_value

    assert aggregate_record[
        "ground_truth_loaded"
    ] is False
    assert aggregate_record[
        "sample_id_sent_to_model"
    ] is False
    assert aggregate_record["files_attached"] is False
    assert aggregate_record[
        "auto_approval_used"
    ] is False
    assert aggregate_record[
        "maximum_model_output_tokens_supplied"
    ] is False

    aggregate_records.append(aggregate_record)

    checkpoint_inventory.append(
        {
            "request_plan_position": plan_entry[
                "request_plan_position"
            ],
            "record_index": plan_entry[
                "record_index"
            ],
            "sample_id": plan_entry["sample_id"],
            "condition": plan_entry["condition"],
            "relative_path": project_relative_path(
                checkpoint_path
            ),
            "sha256": checkpoint_sha256,
        }
    )


assert len(aggregate_records) == 400

assert len(
    {
        (
            record["record_index"],
            record["condition"],
        )
        for record in aggregate_records
    }
) == 400

assert len(
    {
        record["sample_id"]
        for record in aggregate_records
    }
) == 200

assert Counter(
    record["condition"]
    for record in aggregate_records
) == Counter(
    {
        "structured": 200,
        "deterministic_text": 200,
    }
)


results_jsonl_bytes = "".join(
    json.dumps(
        record,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )
    + "\n"
    for record in aggregate_records
).encode("utf-8")

results_jsonl_action = (
    write_artifact_atomically_without_overwrite(
        BATCH_RESULTS_JSONL_PATH,
        results_jsonl_bytes,
    )
)

assert (
    BATCH_RESULTS_JSONL_PATH.read_bytes()
    == results_jsonl_bytes
)
assert (
    read_jsonl(BATCH_RESULTS_JSONL_PATH)
    == aggregate_records
)

results_jsonl_sha256 = sha256_file(
    BATCH_RESULTS_JSONL_PATH
)


def count_records(predicate) -> int:
    return sum(
        bool(predicate(record))
        for record in aggregate_records
    )


condition_summaries = {}

for condition in [
    "structured",
    "deterministic_text",
]:
    condition_records = [
        record
        for record in aggregate_records
        if record["condition"] == condition
    ]

    condition_summaries[condition] = {
        "checkpoint_count": len(
            condition_records
        ),
        "request_completed_count": sum(
            record["request_completed"] is True
            for record in condition_records
        ),
        "visible_json_parsed_count": sum(
            record["visible_json_parsed"] is True
            for record in condition_records
        ),
        "schema_structure_valid_count": sum(
            record["schema_structure_valid"] is True
            for record in condition_records
        ),
        "grounding_valid_count": sum(
            record["grounding_valid"] is True
            for record in condition_records
        ),
        "overall_response_valid_count": sum(
            record["overall_response_valid"] is True
            for record in condition_records
        ),
        "invalid_response_count": sum(
            record["overall_response_valid"] is False
            for record in condition_records
        ),
        "timeout_count": sum(
            record["error_type"] == "TimeoutExpired"
            for record in condition_records
        ),
        "budget_exhausted_count": sum(
            record["budget_exhausted"] is True
            for record in condition_records
        ),
    }


records_by_sample_id = {}

for record in aggregate_records:
    records_by_sample_id.setdefault(
        record["sample_id"],
        {},
    )[record["condition"]] = record

assert len(records_by_sample_id) == 200

assert all(
    set(pair) == {
        "structured",
        "deterministic_text",
    }
    for pair in records_by_sample_id.values()
)


valid_conditions_per_pair = Counter(
    sum(
        pair[condition][
            "overall_response_valid"
        ] is True
        for condition in [
            "structured",
            "deterministic_text",
        ]
    )
    for pair in records_by_sample_id.values()
)

paired_availability = {
    "both_conditions_valid_count": (
        valid_conditions_per_pair[2]
    ),
    "exactly_one_condition_valid_count": (
        valid_conditions_per_pair[1]
    ),
    "neither_condition_valid_count": (
        valid_conditions_per_pair[0]
    ),
}

assert sum(paired_availability.values()) == 200


error_type_counts = Counter(
    record["error_type"]
    for record in aggregate_records
    if record["error_type"] is not None
)

checkpoint_origin_counts = Counter(
    record["checkpoint_origin"]
    for record in aggregate_records
)


response_status = {
    "checkpoint_count": len(aggregate_records),
    "request_completed_count": count_records(
        lambda record: (
            record["request_completed"] is True
        )
    ),
    "overall_response_valid_count": count_records(
        lambda record: (
            record["overall_response_valid"] is True
        )
    ),
    "invalid_response_count": count_records(
        lambda record: (
            record["overall_response_valid"] is False
        )
    ),
    "execution_error_count": count_records(
        lambda record: (
            record["error_type"] is not None
        )
    ),
    "error_type_counts": dict(
        sorted(error_type_counts.items())
    ),
    "completed_but_invalid_count": count_records(
        lambda record: (
            record["request_completed"] is True
            and record["overall_response_valid"] is False
        )
    ),
    "schema_invalid_after_json_parse_count": (
        count_records(
            lambda record: (
                record["visible_json_parsed"] is True
                and record[
                    "schema_structure_valid"
                ] is False
            )
        )
    ),
    "budget_exhausted_count": count_records(
        lambda record: (
            record["budget_exhausted"] is True
        )
    ),
    "by_condition": condition_summaries,
}


locked_paths = {
    "structured_jsonl": STRUCTURED_PATH,
    "deterministic_text_jsonl": TEXT_PATH,
    "equivalence_validation_csv": EQUIVALENCE_PATH,
    "representation_manifest": (
        REPRESENTATION_MANIFEST_PATH
    ),
    "llm_protocol": PROTOCOL_PATH,
    "llm_output_schema": OUTPUT_SCHEMA_PATH,
    "reliability_manifest": (
        RELIABILITY_MANIFEST_PATH
    ),
}

locked_artifacts = {
    name: {
        "relative_path": project_relative_path(path),
        "sha256": sha256_file(path),
    }
    for name, path in locked_paths.items()
}


if BATCH_MANIFEST_PATH.exists():
    existing_manifest = json.loads(
        BATCH_MANIFEST_PATH.read_text(
            encoding="utf-8"
        )
    )

    manifest_created_at_utc = (
        existing_manifest.get(
            "created_at_utc"
        )
    )

    if not isinstance(
        manifest_created_at_utc,
        str,
    ):
        raise RuntimeError(
            "Existing manifest has no valid "
            "created_at_utc."
        )
else:
    manifest_created_at_utc = datetime.now(
        timezone.utc
    ).isoformat()


batch_manifest = {
    "manifest_version": "0.1.0",
    "created_at_utc": manifest_created_at_utc,
    "experiment_phase": (
        "formal_batch_inference"
    ),
    "feature_set": {
        "id": "primary_46",
        "feature_count": 46,
    },
    "backend": {
        "name": "opencode",
        "version": OPENCODE_VERSION,
        "model_route": OPENCODE_MODEL_ROUTE,
        "prompt_template_version": (
            OPENCODE_PROMPT_TEMPLATE_VERSION
        ),
        "pure_mode": True,
        "empty_temporary_directory_per_request": True,
        "files_attached": False,
        "auto_approval": False,
        "event_format": "json",
        "subprocess_timeout_seconds": (
            OPENCODE_REQUEST_TIMEOUT_SECONDS
        ),
        "maximum_model_output_tokens_supplied": False,
        "maximum_concurrent_requests": (
            MAXIMUM_CONCURRENT_REQUESTS
        ),
        "automatic_retry_used": False,
    },
    "locked_artifacts": locked_artifacts,
    "request_plan": {
        "paired_sample_count": 200,
        "request_count": 400,
        "structured_request_count": 200,
        "deterministic_text_request_count": 200,
        "structured_first_pair_count": 100,
        "deterministic_text_first_pair_count": 100,
        "reused_reliability_checkpoint_count": 10,
        "sample_id_sent_to_model": False,
    },
    "checkpoint_inventory": checkpoint_inventory,
    "checkpoint_inventory_sha256": (
        stable_json_sha256(
            checkpoint_inventory
        )
    ),
    "checkpoint_origin_counts": dict(
        sorted(checkpoint_origin_counts.items())
    ),
    "response_status": response_status,
    "paired_response_availability": (
        paired_availability
    ),
    "results_artifact": {
        "relative_path": project_relative_path(
            BATCH_RESULTS_JSONL_PATH
        ),
        "record_count": len(
            aggregate_records
        ),
        "byte_count": len(
            results_jsonl_bytes
        ),
        "sha256": results_jsonl_sha256,
        "ordering": (
            "request_plan_position_ascending"
        ),
    },
    "safety": {
        "ground_truth_path_defined": False,
        "ground_truth_loaded": False,
        "credentials_retained": False,
        "hidden_reasoning_retained": False,
        "network_request_made_during_aggregation": False,
    },
}


manifest_bytes = (
    json.dumps(
        batch_manifest,
        ensure_ascii=False,
        sort_keys=True,
        indent=2,
    )
    + "\n"
).encode("utf-8")

manifest_action = (
    write_artifact_atomically_without_overwrite(
        BATCH_MANIFEST_PATH,
        manifest_bytes,
    )
)


persisted_manifest = json.loads(
    BATCH_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

assert persisted_manifest == batch_manifest

assert (
    persisted_manifest[
        "results_artifact"
    ]["sha256"]
    == sha256_file(
        BATCH_RESULTS_JSONL_PATH
    )
)

assert (
    persisted_manifest[
        "results_artifact"
    ]["record_count"]
    == 400
)

assert (
    persisted_manifest[
        "safety"
    ]["ground_truth_loaded"]
    is False
)


aggregation_summary = pd.Series(
    {
        "results_jsonl_action": (
            results_jsonl_action
        ),
        "manifest_action": (
            manifest_action
        ),
        "results_record_count": len(
            aggregate_records
        ),
        "paired_sample_count": len(
            records_by_sample_id
        ),
        "request_completed_count": (
            response_status[
                "request_completed_count"
            ]
        ),
        "overall_response_valid_count": (
            response_status[
                "overall_response_valid_count"
            ]
        ),
        "timeout_count": (
            error_type_counts.get(
                "TimeoutExpired",
                0,
            )
        ),
        "completed_but_invalid_count": (
            response_status[
                "completed_but_invalid_count"
            ]
        ),
        "structured_valid_count": (
            condition_summaries[
                "structured"
            ][
                "overall_response_valid_count"
            ]
        ),
        "deterministic_text_valid_count": (
            condition_summaries[
                "deterministic_text"
            ][
                "overall_response_valid_count"
            ]
        ),
        "both_conditions_valid_pair_count": (
            paired_availability[
                "both_conditions_valid_count"
            ]
        ),
        "exactly_one_condition_valid_pair_count": (
            paired_availability[
                "exactly_one_condition_valid_count"
            ]
        ),
        "neither_condition_valid_pair_count": (
            paired_availability[
                "neither_condition_valid_count"
            ]
        ),
        "results_jsonl_sha256": (
            results_jsonl_sha256
        ),
        "manifest_sha256": sha256_file(
            BATCH_MANIFEST_PATH
        ),
        "all_checkpoint_identities_reverified": True,
        "automatic_retry_used": False,
        "network_request_made_by_this_cell": False,
        "ground_truth_loaded": False,
        "notebook_07_complete": True,
    },
    name="value",
)


display(
    pd.DataFrame.from_dict(
        condition_summaries,
        orient="index",
    )
    .rename_axis("condition")
    .reset_index()
)

aggregation_summary

,condition,checkpoint_count,request_completed_count,visible_json_parsed_count,schema_structure_valid_count,grounding_valid_count,overall_response_valid_count,invalid_response_count,timeout_count,budget_exhausted_count
0,structured,200,179,179,179,179,179,21,21,0
1,deterministic_text,200,181,181,180,180,180,20,19,0


results_jsonl_action                                                                created
manifest_action                                                                     created
results_record_count                                                                    400
paired_sample_count                                                                     200
request_completed_count                                                                 360
overall_response_valid_count                                                            359
timeout_count                                                                            40
completed_but_invalid_count                                                               1
structured_valid_count                                                                  179
deterministic_text_valid_count                                                          180
both_conditions_valid_pair_count                                                

## Notebook 07 summary and handoff

### Purpose

This notebook completed the formal, resumable LLM batch-inference stage for the locked `primary_46` paired experiment. It collected one structured-representation response and one deterministic-text response for each of 200 previously selected network-flow records, producing 400 condition-level observations.

Private ground truth was not loaded, joined, inspected, or inferred anywhere in this notebook.

### Locked inference protocol

All formal requests used the backend configuration authorised in Notebook 06:

- Backend: OpenCode `1.18.22`
- Model route: `uoa/MiniMax-M3`
- Prompt-template version: `0.1.0`
- Feature set: `primary_46`
- OpenCode `--pure` mode
- One empty temporary directory per request
- No file attachments
- No automatic approval
- JSON event output
- 300-second subprocess timeout
- No notebook-supplied model-output token limit
- Maximum of three concurrent requests
- No automatic retry of failed or invalid observations

The request order was fixed before inference. The first representation alternated across records, giving 100 structured-first pairs and 100 deterministic-text-first pairs.

### Reliability-checkpoint reuse

Ten responses previously validated in Notebook 06 were admitted into the formal batch without being requested again.

Before reuse, all ten checkpoints were:

- matched to the current sample, condition, canonical payload, prompt and configuration;
- copied byte-for-byte into the formal checkpoint directory;
- independently reparsed and revalidated using the corrected response validator;
- confirmed to satisfy the locked schema, legal-label, five-citation, distinct-feature, supported-feature and exact observed-value requirements.

No reliability response was regenerated or altered.

### Resumable execution and evidence preservation

The remaining 390 requests were handled by a resumable scheduler. One fixed smoke-test request was executed first, after which the remaining 389 requests were processed with at most three requests in flight.

Each result was written immediately to an atomic per-request checkpoint. Existing checkpoints were validated before being skipped, divergent files were never overwritten, and failed or invalid results were preserved as experimental evidence.

The scheduler submitted all missing requests without a fatal scheduling or persistence error.

### Formal inference outcome

The final checkpoint store contains all 400 planned request keys across all 200 paired samples.

| Outcome | Overall | Structured | Deterministic text |
|---|---:|---:|---:|
| Planned requests | 400 | 200 | 200 |
| Requests completed | 360 | 179 | 181 |
| Overall valid responses | 359 | 179 | 180 |
| Timeout checkpoints | 40 | 21 | 19 |
| Completed but schema-invalid responses | 1 | 0 | 1 |

The overall valid-response coverage was **89.75%** (`359/400`):

- Structured coverage: **89.5%** (`179/200`)
- Deterministic-text coverage: **90.0%** (`180/200`)

All 40 execution errors were preserved as `TimeoutExpired` checkpoints under the locked 300-second timeout. The only completed but invalid response was the deterministic-text condition for `pilot_007`; its visible response parsed as JSON but did not satisfy the locked response schema.

These execution and validation failures are not classification errors and were not assigned replacement predictions.

### Paired-response availability

Across the 200 sampled records:

- 178 records have valid responses under both representation conditions;
- 3 records have a valid response under exactly one condition;
- 19 records have no valid response under either condition.

Thus, 178 complete valid pairs are available for downstream paired comparisons. Any analysis requiring both representations must use this explicitly defined paired-availability set and report the associated coverage.

### Formal artifacts

The ordered checkpoint evidence was aggregated into:

- `results/inference/primary_46/opencode_batch_results.jsonl`
- `results/inference/primary_46/opencode_batch_manifest.json`

The results JSONL contains 400 records ordered by `request_plan_position`. The manifest records the locked input hashes, backend configuration, request-plan counts, individual checkpoint inventory and hashes, response-status counts, paired availability, safety state, and the exact SHA-256 digest of the aggregated results artifact.

The complete results and manifest SHA-256 values are reported in the final aggregation output above and preserved in the manifest. Neither artifact is silently overwritten if divergent content already exists.

### Interpretation boundary

This notebook reports inference execution and response-validity outcomes only. It does not measure detection accuracy because private labels remained inaccessible.

In particular:

- timeout and schema-invalid observations must not be treated as predicted labels;
- valid feature citations establish input grounding, not causal or objective explanation correctness;
- failed or invalid responses must not be repaired after labels become available;
- the previously documented sample date-distribution confounding remains a study limitation, and the sample was not redrawn.

### Handoff to Notebook 08

Notebook 08 will independently verify the aggregated artifacts, checkpoint inventory, hashes, paired coverage, execution state, JSON/schema validity, legal labels, citation count and distinctness, supported feature names, and exact observed-value grounding.

It will then freeze the inference dataset for downstream analysis while continuing to prohibit access to private ground truth.

Private ground truth will first become accessible in Notebook 09, after the inference results have been independently validated and frozen.